In [3]:
!pip install tensorflow_privacy -q

In [1]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, cohen_kappa_score, matthews_corrcoef
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, classification_report, confusion_matrix, hamming_loss
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from dataclasses import dataclass
from typing import List, Tuple, Dict
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from torch.nn import functional as F
from opacus import PrivacyEngine
import scienceplots
import pathlib
from tabulate import tabulate
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.models import Model
import joblib
import psutil


# NEW: Configure TensorFlow GPU settings and check environment
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"TensorFlow detected {len(physical_devices)} GPU(s): {physical_devices}")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}. Falling back to CPU.")
        os.environ['CUDA_VISIBLE_DEVICES'] = ''  # Disable GPU
else:
    print("No GPU detected. Running TensorFlow on CPU.")
print(f"TensorFlow version: {tf.__version__}")

plt.style.use(["science", "no-latex"])

@dataclass
class Parameters:
    lr: float = 0.01
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sample_rate: float = 0.2
    noise_multiplier: float = 1.5
    max_grad_norm: float = 0.5
    delta: float = 1e-4
    epochs: int = 100
    num_clients: int = 5
    batch_size: int = 128
    communication_rounds: int = 10
    conv_filters: List[int] = None
    kernel_size: int = 3
    dropout_rate: float = 0.3
    use_smote: bool = False
    use_dp: bool = True
    model_type: str = 'dl' # 'ml', 'dl', or 'both'
    multi_output: bool = True  # New flag for multi-output vs single-output

args = Parameters(conv_filters=[32, 64])
os.makedirs('classification_reports', exist_ok=True)
os.makedirs('confusion_matrices', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Sampling functions (unchanged)
def sample_50_percent_by_label(df):
    if 'Label' not in df.columns:
        print("Error: 'Label' column not found in DataFrame for sampling")
        return df
    print("Initial Label value counts before sampling:")
    print(df['Label'].value_counts())
    sampled_df = df.groupby('Label', group_keys=False).apply(
        lambda x: x.sample(frac=0.4, random_state=42)
    ).reset_index(drop=True)
    print("\nSampled Label value counts (50% of each):")
    print(sampled_df['Label'].value_counts())
    print(f"\nOriginal rows: {len(df)}, Sampled rows: {len(sampled_df)}")
    return sampled_df

def sample_50_percent_by_traffic_type(df):
    if 'Traffic Type' not in df.columns:
        print("Error: 'Traffic Type' column not found in DataFrame for sampling")
        return df
    print("Initial Traffic Type value counts before sampling:")
    print(df['Traffic Type'].value_counts())
    sampled_df = df.groupby('Traffic Type', group_keys=False).apply(
        lambda x: x.sample(frac=0.3, random_state=42)
    ).reset_index(drop=True)
    print("\nSampled Label value counts (50% of each):")
    print(sampled_df['Traffic Type'].value_counts())
    print(f"\nOriginal rows: {len(df)}, Sampled rows: {len(sampled_df)}")
    return sampled_df

def sample_by_label_with_thresholds(df, threshold_high=8000, threshold_low=4000):
    if 'Label' not in df.columns:
        print("Error: 'Label' column not found in DataFrame for sampling")
        return df
    print("Initial Label value counts before sampling:")
    print(df['Label'].value_counts())
    def custom_sample(group):
        group_size = len(group)
        if group_size > threshold_high:
            sampled = group.sample(frac=0.4, random_state=42)
        elif group_size > threshold_low:
            sampled = group.sample(frac=0.6, random_state=42)
        else:
            sampled = group
        return sampled
    sampled_df = df.groupby('Label', group_keys=False).apply(custom_sample).reset_index(drop=True)
    return sampled_df

def sample_by_Traffic_Type_with_thresholds(df, threshold_high=8000, threshold_low=4000):
    if 'Traffic Type' not in df.columns:
        print("Error: 'Traffic Type' column not found in DataFrame for sampling")
        return df
    print("Initial Traffic Type value counts before sampling:")
    print(df['Traffic Type'].value_counts())
    def custom_sample(group):
        group_size = len(group)
        if group_size > threshold_high:
            sampled = group.sample(frac=0.4, random_state=42)
        elif group_size > threshold_low:
            sampled = group.sample(frac=0.6, random_state=42)
        else:
            sampled = group
        return sampled
    sampled_df = df.groupby('Traffic Type', group_keys=False).apply(custom_sample).reset_index(drop=True)
    return sampled_df

def sample_40_percent_by_label_and_traffic(df):
    if 'Label' not in df.columns or 'Traffic Type' not in df.columns:
        missing_cols = [col for col in ['Label', 'Traffic Type'] if col not in df.columns]
        print(f"Error: {missing_cols} column(s) not found in DataFrame for sampling")
        return df
    print("Initial Label and Traffic value counts before sampling:")
    print(df.groupby(['Label', 'Traffic Type']).size().to_string())
    sampled_df = df.groupby(['Label', 'Traffic Type'], group_keys=False).apply(
        lambda x: x.sample(frac=0.4, random_state=42)
    ).reset_index(drop=True)
    print("\nSampled Label and Traffic value counts (40% of each):")
    print(sampled_df.groupby(['Label', 'Traffic Type']).size().to_string())
    print(f"\nOriginal rows: {len(df)}, Sampled rows: {len(sampled_df)}")
    return sampled_df

def sample_by_label_and_traffic_with_limits(df, min_samples=3000, max_samples=3500):
    if 'Label' not in df.columns or 'Traffic Type' not in df.columns:
        missing_cols = [col for col in ['Label', 'Traffic Type'] if col in df.columns]
        print(f"Error: {missing_cols} column(s) not found in DataFrame")
        return df
    if not isinstance(min_samples, int) or not isinstance(max_samples, int):
        print("Error: min_samples and max_samples must be integers")
        return df
    if min_samples < 0 or max_samples < 0:
        print("Error: min_samples and max_samples must be non-negative")
        return df
    if min_samples > max_samples:
        print("Error: min_samples cannot be greater than max_samples")
        return df
    print("Initial Label and Traffic Type counts before sampling:")
    initial_counts = df.groupby(['Label', 'Traffic Type']).size()
    print(initial_counts.to_string())
    def sample_group(group):
        n = len(group)
        if n < min_samples:
            print(f"Warning: Group {group.name} has {n} rows, less than min_samples ({min_samples}). Taking all rows.")
            return group
        elif n > max_samples:
            return group.sample(n=max_samples, random_state=42)
        else:
            return group.sample(n=min_samples, random_state=42)
    sampled_df = df.groupby(['Label', 'Traffic Type'], group_keys=False).apply(sample_group).reset_index(drop=True)
    print("\nSampled Label and Traffic value counts:")
    sampled_counts = sampled_df.groupby(['Label', 'Traffic Type']).size()
    print(sampled_counts.to_string())
    print(f"\nOriginal rows: {len(df)}, Sampled rows: {len(sampled_df)}")
    return sampled_df

cols_to_use = ['ARP', 'BOOTP_hlen', 'BOOTP_options', 'DHCP_options', 'DNS_ancount', 'DNS_arcount', 'DNS_id',
               'DNS_nscount', 'DNS_opcode', 'DNS_qr', 'DNS_ra', 'DNS_rd', 'DNS_z', 'Direction_ratio', 'Entropy_Min',
               'Entropy_Var', 'HTTP_method', 'HTTP_status', 'IAT_mean', 'ICMP_Chksum_Status', 'ICMP_chksum', 'ICMP_seq',
               'ICMP_type', 'IP_Dst_Entropy', 'IP_add_count', 'IP_chksum', 'IP_proto', 'IP_version', 'MQTT_type',
               'Min_Elapsed_Time', 'NTP_Interval', 'Payload_Len_IQR', 'Payload_Len_Q1', 'Payload_Len_Q3', 'Pck_Size_Avg',
               'Pck_Size_Kurt', 'Pck_Size_Max', 'Pck_Size_Med', 'Pck_Size_Min', 'Pck_Size_Q1', 'Pck_Size_Q3',
               'Pck_Size_Sum', 'Pck_Size_Var', 'Pck_size', 'Portcl_dst', 'Portcl_src', 'Protocol_ICMP_Ratio',
               'Protocol_UDP', 'Protocol_UDP_Ratio', 'STUN_method', 'TCP_ACK', 'TCP_CWR', 'TCP_FIN', 'TCP_PSH',
               'TCP_RST', 'TCP_SYN', 'TCP_URG', 'TCP_chksum', 'TCP_dataofs', 'TCP_mss_values', 'TCP_seq', 'TCP_sport',
               'TCP_window_scaling', 'TLS_Cipher_Suites_Len', 'TLS_Server', 'TTL_default', 'Time_Since_Prev_Frame',
               'UDP_chksum', 'UDP_sport', 'dport23', 'dport_bare', 'payload_bytes', 'sport23', 'sport_bare','Label', 'Traffic Type']

def load_and_preprocess_data(csv_path):
    csv_path = pathlib.Path(csv_path)
    df = pd.read_csv(csv_path, usecols=cols_to_use, low_memory=False)
    print(f"Loaded {csv_path.name} with {len(df)} rows and {len(df.columns)} columns.")
    if 'MAC' in df.columns:
        df = df.drop(columns=['MAC'])
    print("\nNull values per column:")
    print(df.isna().sum())
    df = df.fillna(0)
    print(f"After dropping NaN rows, dataset has {len(df)} rows.")
    duplicates = df.duplicated().sum()
    print(f"\nNumber of duplicate rows: {duplicates}")
    if duplicates > 0:
        df = df.drop_duplicates()
        print(f"Dropped {duplicates} duplicate rows.")
    print("\nColumn data types:")
    print(df.dtypes)
    if 'Traffic Type' not in df.columns:
        print("Warning: 'Traffic Type' column not found. Assuming binary attack category based on 'Label'.")
        df['Traffic Type'] = df['Label'].apply(lambda x: 'Attack' if 'attack' in str(x).lower() else 'Benign')
    print("\nDevice type class distribution before dropping rare classes:")
    device_counts = pd.Series(df['Label']).value_counts()
    print(device_counts)
    print("\nAttack category class distribution before dropping rare classes:")
    attack_counts = pd.Series(df['Traffic Type']).value_counts()
    print(attack_counts)
    print("\nCombined (Label, Traffic Type) distribution before dropping rare classes:")
    combined_counts = df.groupby(['Label', 'Traffic Type']).size()
    print(combined_counts)
    min_samples = 3
    rare_device_classes = device_counts[device_counts < min_samples].index
    rare_attack_classes = attack_counts[attack_counts < min_samples].index
    rare_combined = combined_counts[combined_counts < min_samples].index
    print("\nDropping device type classes with <{} samples: {}".format(min_samples, rare_device_classes.tolist()))
    print("Dropping attack category classes with <{} samples: {}".format(min_samples, rare_attack_classes.tolist()))
    print("Dropping combined (Label, Traffic Type) classes with <{} samples: {}".format(min_samples, rare_combined.tolist()))
    df = df[~df['Label'].isin(rare_device_classes)]
    df = df[~df['Traffic Type'].isin(rare_attack_classes)]
    df = df[~df[['Label', 'Traffic Type']].apply(tuple, axis=1).isin(rare_combined)]
    print(f"\nAfter dropping rare classes, dataset has {len(df)} rows.")
    print("\nDevice type class distribution after dropping rare classes:")
    device_counts = pd.Series(df['Label']).value_counts()
    print(device_counts)
    print("\nAttack category class distribution after dropping rare classes:")
    attack_counts = pd.Series(df['Traffic Type']).value_counts()
    print(attack_counts)
    columns_to_drop = ['BOOTP_flags', 'Ether_Src_OUI', 'Ether_Dst_OUI', 'Highest_Layer']
    columns_to_drop = [col for col in columns_to_drop if col in df.columns]
    df = df.drop(columns=columns_to_drop)
    df = sample_50_percent_by_label(df)
    X = df.drop(columns=['Label', 'Traffic Type'])
    y_device = df['Label']
    y_attack = df['Traffic Type']
    non_numeric_cols = X.select_dtypes(include=['object']).columns
    print(f"\nNon-numeric columns: {list(non_numeric_cols)}")
    label_encoders = {}
    for col in non_numeric_cols:
        print(f"Encoding: {col}")
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le
    if not all(X.dtypes.apply(lambda x: np.issubdtype(x, np.number))):
        print("Error: Some columns are non-numeric after encoding:")
        print(X.dtypes)
        raise ValueError("Non-numeric columns detected.")
    if np.any(np.isinf(X.to_numpy())):
        print("Warning: Inf values found in features. Replacing with max finite value.")
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.fillna(X.max().max())
    if args.use_smote:
        print("\nApplying SMOTE to balance classes...")
        if args.multi_output:
            y_combined = np.column_stack([y_device, y_attack])
            smote = SMOTE(random_state=42, k_neighbors=5)
            try:
                X_resampled, y_combined_resampled = smote.fit_resample(X, y_combined)
                y_device_resampled = y_combined_resampled[:, 0]
                y_attack_resampled = y_combined_resampled[:, 1]
                print(f"After SMOTE, dataset has {len(X_resampled)} rows.")
                print("\nDevice type distribution after SMOTE:")
                print(pd.Series(y_device_resampled).value_counts())
                print("\nAttack category distribution after SMOTE:")
                print(pd.Series(y_attack_resampled).value_counts())
            except ValueError as e:
                print(f"SMOTE failed: {e}. Disabling SMOTE.")
                X_resampled = X
                y_device_resampled = y_device
                y_attack_resampled = y_attack
        else:
            smote = SMOTE(random_state=42, k_neighbors=5)
            try:
                X_attack, y_attack_resampled = smote.fit_resample(X, y_attack)
                X_device, y_device_resampled = smote.fit_resample(X, y_device)
                print(f"After SMOTE, attack dataset: {len(X_attack)} rows, device dataset: {len(X_device)} rows.")
                X_resampled = X_attack  # Use attack data as primary; device handled separately
                y_attack_resampled = y_attack_resampled
                y_device_resampled = y_device_resampled
            except ValueError as e:
                print(f"SMOTE failed: {e}. Disabling SMOTE.")
                X_resampled = X
                y_attack_resampled = y_attack
                y_device_resampled = y_device
    else:
        print("\nSMOTE disabled. Using original dataset.")
        X_resampled = X
        y_device_resampled = y_device
        y_attack_resampled = y_attack
        print(f"Dataset has {len(X_resampled)} rows.")
    scaler = StandardScaler()
    variances = X_resampled.var()
    zero_var_cols = variances[variances == 0].index
    if len(zero_var_cols) > 0:
        print(f"Removing {len(zero_var_cols)} zero-variance features: {zero_var_cols.tolist()}")
        X_resampled = X_resampled.drop(columns=zero_var_cols)
    X_scaled = scaler.fit_transform(X_resampled)
    if np.any(np.isnan(X_scaled)) or np.any(np.isinf(X_scaled)):
        print("Error: NaN or inf in X_scaled after scaling.")
        X_scaled = np.nan_to_num(X_scaled, nan=np.nanmean(X_scaled), posinf=X_scaled.max(), neginf=X_scaled.min())
    le_device = LabelEncoder()
    le_attack = LabelEncoder()
    y_device_encoded = le_device.fit_transform(y_device_resampled)
    y_attack_encoded = le_attack.fit_transform(y_attack_resampled)
    return X_scaled, y_device_encoded, y_attack_encoded, le_device, le_attack, X_resampled.columns.tolist()

def compute_metrics(y_true, y_pred, label_type, y_probs=None, label_encoder=None):
    # Ensure y_true and y_pred only contain valid classes from label_encoder
    if label_encoder is not None:
        valid_classes = np.arange(len(label_encoder.classes_))
        valid_mask = np.isin(y_true, valid_classes) & np.isin(y_pred, valid_classes)
        y_true = y_true[valid_mask]
        y_pred = y_pred[valid_mask]
        if y_probs is not None:
            y_probs = y_probs[valid_mask]

    cm = confusion_matrix(y_true, y_pred, labels=valid_classes if label_encoder else None)
    tn = np.sum(cm) - (np.sum(cm, axis=0) + np.sum(cm, axis=1) - np.diag(cm))
    fp = np.sum(cm, axis=0) - np.diag(cm)
    fn = np.sum(cm, axis=1) - np.diag(cm)
    tp = np.diag(cm)
    tnr = np.divide(tn, (tn + fp), out=np.zeros_like(tn, dtype=float), where=(tn + fp) != 0)
    specificity = tnr
    tpr = np.divide(tp, (tp + fn), out=np.zeros_like(tp, dtype=float), where=(tp + fn) != 0)
    fpr = np.divide(fp, (fp + tn), out=np.zeros_like(fp, dtype=float), where=(fp + tn) != 0)
    fnr = np.divide(fn, (tp + fn), out=np.zeros_like(fn, dtype=float), where=(tp + fn) != 0)
    per_class_accuracy = np.divide(tp, np.sum(cm, axis=1), out=np.zeros_like(tp, dtype=float), where=np.sum(cm, axis=1) != 0)
    hamming = hamming_loss(y_true, y_pred)
    metrics = {
        f'{label_type}_precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        f'{label_type}_recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        f'{label_type}_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        f'{label_type}_kappa': cohen_kappa_score(y_true, y_pred),
        f'{label_type}_mcc': matthews_corrcoef(y_true, y_pred),
        f'{label_type}_balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        f'{label_type}_hamming_loss': hamming,
        f'{label_type}_tnr': np.mean(tnr),
        f'{label_type}_specificity': np.mean(specificity),
        f'{label_type}_tpr': np.mean(tpr),
        f'{label_type}_fpr': np.mean(fpr),
        f'{label_type}_fnr': np.mean(fnr)
    }
    if y_probs is not None:
        unique_classes = np.unique(y_true)
        expected_n_classes = len(label_encoder.classes_) if label_encoder else len(unique_classes)
        print(f"{label_type} unique classes: {unique_classes}")
        print(f"{label_type} y_probs shape: {y_probs.shape}")
        print(f"{label_type} expected classes: {expected_n_classes}")
        if y_probs.shape[1] != expected_n_classes:
            print(f"Warning: y_probs shape mismatch for {label_type}: got {y_probs.shape[1]}, expected {expected_n_classes}")
            metrics[f'{label_type}_auc'] = np.nan
        elif len(unique_classes) >= 2:
            try:
                if len(unique_classes) == 2:
                    pos_class_idx = np.max(unique_classes)
                    y_true_binary = (y_true == pos_class_idx).astype(int)
                    metrics[f'{label_type}_auc'] = roc_auc_score(y_true_binary, y_probs[:, pos_class_idx])
                    print(f"{label_type} AUC (binary): {metrics[f'{label_type}_auc']}")
                else:
                    metrics[f'{label_type}_auc'] = roc_auc_score(y_true, y_probs, multi_class='ovr')
                    print(f"{label_type} AUC (multiclass): {metrics[f'{label_type}_auc']}")
            except Exception as e:
                print(f"Warning: AUC calculation failed for {label_type}: {e}")
                metrics[f'{label_type}_auc'] = np.nan
        else:
            print(f"Warning: AUC calculation skipped for {label_type}: Only one class present")
            metrics[f'{label_type}_auc'] = np.nan
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    if label_encoder is not None:
        class_names = label_encoder.classes_
        report_dict = {}
        for i, label in enumerate(class_names):
            if str(i) in report:
                report_dict[label] = report[str(i)]
                report_dict[label]['accuracy'] = per_class_accuracy[i] if i < len(per_class_accuracy) else 0.0
        for key in ['accuracy', 'macro avg', 'weighted avg']:
            if key in report:
                report_dict[key] = report[key]
        report = report_dict
    report_df = pd.DataFrame(report).transpose()
    return metrics, report_df, cm

class DatasetLoader:
    def __init__(self, X_scaled, y_device_encoded, y_attack_encoded, le_device, le_attack, num_clients, sample_rate):
        self._num_clients = num_clients
        self._sample_rate = sample_rate
        self._label_encoder_traffic = le_attack
        self._label_encoder_device = le_device
        self._scaler = StandardScaler()
        if not (len(X_scaled) == len(y_device_encoded) == len(y_attack_encoded)):
            raise ValueError(
                f"Inconsistent input sizes: X_scaled={len(X_scaled)}, "
                f"y_device_encoded={len(y_device_encoded)}, "
                f"y_attack_encoded={len(y_attack_encoded)}"
            )
        self._X_train, self._y_train_traffic, self._y_train_device, self._X_test, self._y_test_traffic, self._y_test_device = self._load_train_test_data(
            X_scaled, y_attack_encoded, y_device_encoded
        )
        self._clients_data = self._assign_data_to_clients(self._X_train, self._y_train_traffic, self._y_train_device)

    @property
    def clients_data(self):
        return self._clients_data

    @property
    def scaler(self):
        return self._scaler

    @property
    def label_encoders(self):
        return {'traffic': self._label_encoder_traffic, 'device': self._label_encoder_device}

    def get_train_data(self):
        train_dataset = TensorDataset(
            torch.tensor(self._X_train, dtype=torch.float32).unsqueeze(1),
            torch.stack([torch.tensor(self._y_train_traffic, dtype=torch.long), torch.tensor(self._y_train_device, dtype=torch.long)], dim=1)
        )
        return DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())

    def get_test_data(self):
        test_dataset = TensorDataset(
            torch.tensor(self._X_test, dtype=torch.float32).unsqueeze(1),
            torch.stack([torch.tensor(self._y_test_traffic, dtype=torch.long), torch.tensor(self._y_test_device, dtype=torch.long)], dim=1)
        )
        return DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    def get_ml_data(self):
        return self._X_train, self._y_train_traffic, self._y_train_device, self._X_test, self._y_test_traffic, self._y_test_device

    def _load_train_test_data(self, X_scaled, y_attack_encoded, y_device_encoded):
        stratified_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        try:
            for train_idx, test_idx in stratified_split.split(X_scaled, y_attack_encoded):
                X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
                y_train_traffic, y_test_traffic = y_attack_encoded[train_idx], y_attack_encoded[test_idx]
                y_train_device, y_test_device = y_device_encoded[train_idx], y_device_encoded[test_idx]
        except ValueError as e:
            print(f"Stratified split failed: {e}. Falling back to random split.")
            indices = np.arange(len(X_scaled))
            np.random.shuffle(indices)
            split = int(0.8 * len(X_scaled))
            train_idx, test_idx = indices[:split], indices[split:]
            X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
            y_train_traffic, y_test_traffic = y_attack_encoded[train_idx], y_attack_encoded[test_idx]
            y_train_device, y_test_device = y_device_encoded[train_idx], y_device_encoded[test_idx]
        self._scaler.fit(X_train)
        X_train = self._scaler.transform(X_train)
        X_test = self._scaler.transform(X_test)
        return X_train, y_train_traffic, y_train_device, X_test, y_test_traffic, y_test_device

    def _assign_data_to_clients(self, X_train, y_train_traffic, y_train_device):
        clients_all_data = {}
        num_samples = X_train.shape[0]
        indices = np.arange(num_samples)
        np.random.seed(42)
        np.random.shuffle(indices)
        split_indices = np.array_split(indices, self._num_clients)
        for i, idx in enumerate(split_indices):
            batched_x = X_train[idx]
            batched_y_traffic = y_train_traffic[idx]
            batched_y_device = y_train_device[idx]
            x_tensor = torch.tensor(batched_x, dtype=torch.float32).unsqueeze(1)
            y_traffic_tensor = torch.tensor(batched_y_traffic, dtype=torch.long)
            y_device_tensor = torch.tensor(batched_y_device, dtype=torch.long)
            if torch.any(torch.isnan(x_tensor)) or torch.any(torch.isinf(x_tensor)):
                print(f"Warning: NaN or inf in client_{i} input data.")
                x_tensor = torch.nan_to_num(x_tensor, nan=0.0, posinf=1.0, neginf=-1.0)
            client_dataset = TensorDataset(
                x_tensor,
                torch.stack([y_traffic_tensor, y_device_tensor], dim=1)
            )
            data_loader = DataLoader(
                client_dataset,
                batch_size=args.batch_size,
                shuffle=True,
                num_workers=0,
                pin_memory=torch.cuda.is_available()
            )
            clients_all_data[f'client_{i}'] = {
                'dl_loader': data_loader,
                'X': batched_x,
                'y_traffic': batched_y_traffic,
                'y_device': batched_y_device
            }
        return clients_all_data

    def print_client_label_distribution(self):
        for client_name, data in self._clients_data.items():
            traffic_labels = data['y_traffic'].tolist()
            device_labels = data['y_device'].tolist()
            traffic_counts = {}
            device_counts = {}
            for label in traffic_labels:
                original_label = self._label_encoder_traffic.inverse_transform([label])[0]
                traffic_counts[original_label] = traffic_counts.get(original_label, 0) + 1
            for label in device_labels:
                original_label = self._label_encoder_device.inverse_transform([label])[0]
                device_counts[original_label] = device_counts.get(original_label, 0) + 1
            print(f"\n{client_name} has {len(traffic_labels)} samples:")
            print("Traffic Type distribution:")
            for label_name, count in traffic_counts.items():
                print(f"  {label_name}: {count}")
            print("Device Label distribution:")
            for label_name, count in device_counts.items():
                print(f"  {label_name}: {count}")

class MultiOutputCNN(nn.Module):
    def __init__(self, input_shape, num_classes_traffic, num_classes_device, task='both'):
        super().__init__()
        self.task = task
        self.conv1 = nn.Conv1d(1, args.conv_filters[0], kernel_size=args.kernel_size, padding=args.kernel_size//2)
        self.norm1 = nn.GroupNorm(num_groups=4, num_channels=args.conv_filters[0])
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.conv2 = nn.Conv1d(args.conv_filters[0], args.conv_filters[1], kernel_size=args.kernel_size, padding=args.kernel_size//2)
        self.norm2 = nn.GroupNorm(num_groups=8, num_channels=args.conv_filters[1])
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(args.conv_filters[1] * (input_shape // 4), 128)
        self.relu3 = nn.ReLU()
        self.dropout = nn.Dropout(p=args.dropout_rate)
        if task == 'both':
            self.fc_traffic = nn.Linear(128, num_classes_traffic)
            self.fc_device = nn.Linear(128, num_classes_device)
        elif task == 'traffic':
            self.fc_traffic = nn.Linear(128, num_classes_traffic)
        elif task == 'device':
            self.fc_device = nn.Linear(128, num_classes_device)
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight.data, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.conv1(x)
        x = self.norm1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.norm2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.dropout(x)
        if self.task == 'both':
            traffic_out = self.fc_traffic(x)
            device_out = self.fc_device(x)
            traffic_out = torch.clamp(traffic_out, min=-100, max=100)
            device_out = torch.clamp(device_out, min=-100, max=100)
            return traffic_out, device_out
        elif self.task == 'traffic':
            traffic_out = self.fc_traffic(x)
            return traffic_out
        elif self.task == 'device':
            device_out = self.fc_device(x)
            return device_out

class LSTMModel(nn.Module):
    def __init__(self, input_shape, num_classes_traffic, num_classes_device, task='both'):
        super().__init__()
        self.task = task
        self.lstm = nn.LSTM(input_size=input_shape, hidden_size=64, num_layers=2, batch_first=True)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=args.dropout_rate)
        if task == 'both':
            self.fc_traffic = nn.Linear(128, num_classes_traffic)
            self.fc_device = nn.Linear(128, num_classes_device)
        elif task == 'traffic':
            self.fc_traffic = nn.Linear(128, num_classes_traffic)
        elif task == 'device':
            self.fc_device = nn.Linear(128, num_classes_device)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight.data, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = x.squeeze(1)
        x = x.unsqueeze(1)
        out, _ = self.lstm(x)
        x = self.flatten(out[:, -1, :])
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        if self.task == 'both':
            traffic_out = self.fc_traffic(x)
            device_out = self.fc_device(x)
            return traffic_out, device_out
        elif self.task == 'traffic':
            return self.fc_traffic(x)
        elif self.task == 'device':
            return self.fc_device(x)

class BiLSTM(nn.Module):
    def __init__(self, input_shape, num_classes_traffic, num_classes_device, task='both'):
        super().__init__()
        self.task = task
        self.bilstm = nn.LSTM(input_size=input_shape, hidden_size=64, num_layers=2, batch_first=True, bidirectional=True)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 2, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=args.dropout_rate)
        if task == 'both':
            self.fc_traffic = nn.Linear(128, num_classes_traffic)
            self.fc_device = nn.Linear(128, num_classes_device)
        elif task == 'traffic':
            self.fc_traffic = nn.Linear(128, num_classes_traffic)
        elif task == 'device':
            self.fc_device = nn.Linear(128, num_classes_device)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight.data, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = x.squeeze(1)
        x = x.unsqueeze(1)
        out, _ = self.bilstm(x)
        x = self.flatten(out[:, -1, :])
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        if self.task == 'both':
            traffic_out = self.fc_traffic(x)
            device_out = self.fc_device(x)
            return traffic_out, device_out
        elif self.task == 'traffic':
            return self.fc_traffic(x)
        elif self.task == 'device':
            return self.fc_device(x)

class TF1DCNN:
    def __init__(self, input_shape, num_classes_device, num_classes_attack, task='both'):
        # NEW: Configure GPU memory growth
        physical_devices = tf.config.list_physical_devices('GPU')
        if physical_devices:
            try:
                for gpu in physical_devices:
                    tf.config.experimental.set_memory_growth(gpu, True)
            except RuntimeError as e:
                print(f"GPU memory configuration failed: {e}")

        self.task = task
        self.model = self._build_1dcnn((input_shape, 1), num_classes_device, num_classes_attack)
        self.input_shape = input_shape

    def _build_1dcnn(self, input_shape, num_classes_device, num_classes_attack):
        inputs = Input(shape=input_shape)
        x = Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
        x = MaxPooling1D(pool_size=2)(x)
        x = Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
        x = MaxPooling1D(pool_size=2)(x)
        x = Flatten()(x)
        x = Dense(128, activation='relu')(x)
        x = Dropout(0.5)(x)
        if self.task == 'both':
            device_output = Dense(num_classes_device, activation='softmax', name='device')(x)
            attack_output = Dense(num_classes_attack, activation='softmax', name='attack')(x)
            outputs = [device_output, attack_output]
        elif self.task == 'traffic':
            attack_output = Dense(num_classes_attack, activation='softmax', name='attack')(x)
            outputs = attack_output
        elif self.task == 'device':
            device_output = Dense(num_classes_device, activation='softmax', name='device')(x)
            outputs = device_output
        model = Model(inputs=inputs, outputs=outputs)
        if self.task == 'both':
            model.compile(
                optimizer='adam',
                loss={'device': 'sparse_categorical_crossentropy', 'attack': 'sparse_categorical_crossentropy'},
                metrics={'device': 'accuracy', 'attack': 'accuracy'}
            )
        else:
            model.compile(
                optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )
        return model

    def get_weights(self):
        return self.model.get_weights()

    def set_weights(self, weights):
        self.model.set_weights(weights)

    def predict(self, x):
        x = x.reshape(-1, self.input_shape, 1)
        if self.task == 'both':
            device_probs, attack_probs = self.model.predict(x, verbose=0)
            device_preds = np.argmax(device_probs, axis=1)
            attack_preds = np.argmax(attack_probs, axis=1)
            return attack_preds, device_preds, attack_probs, device_probs
        elif self.task == 'traffic':
            attack_probs = self.model.predict(x, verbose=0)
            attack_preds = np.argmax(attack_probs, axis=1)
            return attack_preds, None, attack_probs, None
        elif self.task == 'device':
            device_probs = self.model.predict(x, verbose=0)
            device_preds = np.argmax(device_probs, axis=1)
            return None, device_preds, None, device_probs

    def fit(self, X, y_traffic, y_device, epochs, batch_size):
        X = X.reshape(-1, self.input_shape, 1)
        if self.task == 'both':
            self.model.fit(
                X, {'device': y_device, 'attack': y_traffic},
                epochs=epochs, batch_size=batch_size, verbose=1
            )
        elif self.task == 'traffic':
            self.model.fit(X, y_traffic, epochs=epochs, batch_size=batch_size, verbose=1)
        elif self.task == 'device':
            self.model.fit(X, y_device, epochs=epochs, batch_size=batch_size, verbose=1)

class MLModel:
    def __init__(self, model_type, n_estimators=50, multi_output=True):
        self.multi_output = multi_output
        # Extract base model type by removing '_traffic' or '_device' suffixes
        base_model_type = model_type.replace('_traffic', '').replace('_device', '')
        if base_model_type == 'xgboost':
            base_model = XGBClassifier(n_estimators=n_estimators, random_state=42)
        elif base_model_type == 'rf':
            base_model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
        elif base_model_type == 'dt':
            base_model = DecisionTreeClassifier(random_state=42)
        else:
            raise ValueError(f"Unknown ML model type: {base_model_type}")
        if multi_output:
            self.model = MultiOutputClassifier(base_model, n_jobs=-1)
        else:
            self.model = base_model
        self.model_type = model_type

    def fit(self, X, y_traffic, y_device=None):
        if self.multi_output:
            y = np.column_stack((y_traffic, y_device))
            self.model.fit(X, y)
        else:
            self.model.fit(X, y_traffic)

    def predict(self, X):
        if self.multi_output:
            preds = self.model.predict(X)
            probs = self.model.predict_proba(X)
            attack_preds = preds[:, 0]
            device_preds = preds[:, 1]
            attack_probs = probs[0]
            device_probs = probs[1]
            return attack_preds, device_preds, attack_probs, device_probs
        else:
            preds = self.model.predict(X)
            probs = self.model.predict_proba(X)
            # Return None for device if single-output model
            return preds, None, probs, None

class MLEnsembleModel:
    def __init__(self, client_models, model_type, multi_output=True):
        self.client_models = client_models
        self.model_type = model_type
        self.multi_output = multi_output

    def predict(self, X):
        if self.multi_output:
            attack_probs_list = []
            device_probs_list = []
            for model in self.client_models:
                _, _, attack_probs, device_probs = model.predict(X)
                attack_probs_list.append(attack_probs)
                device_probs_list.append(device_probs)
            attack_probs = np.mean(attack_probs_list, axis=0)
            device_probs = np.mean(device_probs_list, axis=0)
            attack_preds = np.argmax(attack_probs, axis=1)
            device_preds = np.argmax(device_probs, axis=1)
            return attack_preds, device_preds, attack_probs, device_probs
        else:
            probs_list = []
            for model in self.client_models:
                _, _, probs, _ = model.predict(X)
                probs_list.append(probs)
            probs = np.mean(probs_list, axis=0)
            preds = np.argmax(probs, axis=1)
            return preds, None, probs, None

class DLServer:
    def __init__(self, model_class, input_size, num_classes_traffic, num_classes_device, task='both'):
        self._model = model_class(input_shape=input_size, num_classes_traffic=num_classes_traffic, num_classes_device=num_classes_device, task=task).to(args.device)
        self._client_parameters = []
        self._client_samples = []

    def add_client(self, parameters, num_samples):
        print(f"Adding client parameters with {num_samples} samples")
        self._client_parameters.append([p.cpu() for p in parameters])  # Ensure parameters are on CPU to save memory
        self._client_samples.append(num_samples)
        print(f"Current number of client parameter sets: {len(self._client_parameters)}")

    def aggregate(self):
        if not self._client_parameters:
            print("Warning: No client parameters to aggregate. Skipping aggregation.")
            return
        print(f"Aggregating {len(self._client_parameters)} client parameter sets")
        weights = [n / sum(self._client_samples) for n in self._client_samples]
        aggregated_params = [torch.zeros_like(p) for p in self._client_parameters[0]]
        for i, params in enumerate(self._client_parameters):
            for j, param in enumerate(params):
                aggregated_params[j] += weights[i] * param
        with torch.no_grad():
            for param, agg_param in zip(self._model.parameters(), aggregated_params):
                param.copy_(agg_param)
        self._client_parameters = []  # Clear after aggregation
        self._client_samples = []  # Clear sample counts
        print("Aggregation completed and client parameters cleared")

    def get_model(self):
        return self._model

class TF1Server:
    def __init__(self, input_shape, num_classes_traffic, num_classes_device, task='both'):
        self._model = TF1DCNN(input_shape, num_classes_device, num_classes_traffic, task=task)
        self._client_weights = []
        self._client_samples = []
        self.task = task

    def add_client(self, weights, num_samples):
        self._client_weights.append(weights)
        self._client_samples.append(num_samples)

    def aggregate(self):
        weights = [n / sum(self._client_samples) for n in self._client_samples]
        aggregated_weights = []
        for layer in zip(*self._client_weights):
            weighted_sum = sum(w * layer for w, layer in zip(weights, layer))
            aggregated_weights.append(weighted_sum)
        self._model.set_weights(aggregated_weights)
        self._client_weights.clear()
        self._client_samples.clear()

    def get_model(self):
        return self._model

class MLServer:
    def __init__(self, model_type, multi_output=True):
        self._models = {}
        self.model_type = model_type
        self.multi_output = multi_output
        self.ensemble_model = None

    def add_client(self, model, client_name, X, y_traffic, y_device=None):
        self._models[client_name] = (model, X, y_traffic, y_device)

    def aggregate(self, X_test):
        # Create ensemble model from client models
        client_models = [model for model, _, _, _ in self._models.values()]
        if not client_models:
            raise ValueError("No client models available for aggregation")
        self.ensemble_model = MLEnsembleModel(client_models, self.model_type, self.multi_output)
        # Return predictions from the ensemble model
        return self.ensemble_model.predict(X_test)

    def get_model(self, client_name=None):
        if client_name:
            return self._models[client_name][0]
        if self.ensemble_model is None:
            raise ValueError("Ensemble model not initialized. Call aggregate() first.")
        return self.ensemble_model

class DLClient:
    def __init__(self, model_class, data_loader, input_size, num_classes_traffic, num_classes_device, y_train_traffic, y_train_device, task='both'):
        self._model = model_class(input_shape=input_size, num_classes_traffic=num_classes_traffic, num_classes_device=num_classes_device, task=task).to(args.device)
        self._optimizer = optim.SGD(self._model.parameters(), lr=args.lr)
        self.data_loader = data_loader
        self.task = task
        try:
            unique_traffic = np.unique(y_train_traffic)
            unique_device = np.unique(y_train_device)
            class_weights_traffic = compute_class_weight('balanced', classes=unique_traffic, y=y_train_traffic)
            class_weights_device = compute_class_weight('balanced', classes=unique_device, y=y_train_device)
            if np.any(np.isnan(class_weights_traffic)) or np.any(np.isinf(class_weights_traffic)):
                class_weights_traffic = np.ones(len(unique_traffic))
            if np.any(np.isnan(class_weights_device)) or np.any(np.isinf(class_weights_device)):
                class_weights_device = np.ones(len(unique_device))
        except:
            class_weights_traffic = np.ones(num_classes_traffic)
            class_weights_device = np.ones(num_classes_device)
        self._loss_traffic = nn.CrossEntropyLoss(weight=torch.tensor(class_weights_traffic, dtype=torch.float32).to(args.device))
        self._loss_device = nn.CrossEntropyLoss(weight=torch.tensor(class_weights_device, dtype=torch.float32).to(args.device))
        if args.use_dp:
            self.privacy_engine = PrivacyEngine()
            try:
                self._model, self._optimizer, self.data_loader = self.privacy_engine.make_private(
                    module=self._model,
                    optimizer=self._optimizer,
                    data_loader=self.data_loader,
                    noise_multiplier=args.noise_multiplier,
                    max_grad_norm=args.max_grad_norm
                )
            except Exception as e:
                print(f"PrivacyEngine failed: {e}. Disabling DP.")
                args.use_dp = False

    def train(self, epochs, client_name):
        self._model.train()
        process = psutil.Process()
        epsilons = []
        memory_usages = []
        cpu_usages = []
        energy_usages = []
        for epoch in range(epochs):
            losses_traffic = []
            losses_device = []
            epoch_start_time = time.time()
            cpu_percent = []
            for batch_idx, (x, y) in enumerate(tqdm(self.data_loader, desc=f'Training {client_name} Epoch {epoch+1}')):
                x = x.to(args.device).to(torch.float32)
                if self.task == 'both':
                    y_traffic, y_device = y[:, 0].to(args.device), y[:, 1].to(args.device)
                    self._optimizer.zero_grad()
                    traffic_out, device_out = self._model(x)
                    loss_traffic = self._loss_traffic(traffic_out, y_traffic)
                    loss_device = self._loss_device(device_out, y_device)
                    loss = loss_traffic + loss_device
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self._model.parameters(), max_norm=1.0)
                    self._optimizer.step()
                    losses_traffic.append(loss_traffic.item())
                    losses_device.append(loss_device.item())
                else:
                    y_target = y[:, 0 if self.task == 'traffic' else 1].to(args.device)
                    self._optimizer.zero_grad()
                    out = self._model(x)
                    loss = self._loss_traffic(out, y_target) if self.task == 'traffic' else self._loss_device(out, y_target)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self._model.parameters(), max_norm=1.0)
                    self._optimizer.step()
                    (losses_traffic if self.task == 'traffic' else losses_device).append(loss.item())
                cpu_percent.append(psutil.cpu_percent(interval=None))
            memory_usage = process.memory_info().rss / (1024 ** 2)  # MB
            cpu_usage = np.mean(cpu_percent) if cpu_percent else 0.0
            epoch_time = time.time() - epoch_start_time
            energy_usage = cpu_usage * epoch_time  # Proxy: CPU% * time (seconds)
            memory_usages.append(memory_usage)
            cpu_usages.append(cpu_usage)
            energy_usages.append(energy_usage)
            epsilon = self.privacy_engine.get_epsilon(args.delta) if args.use_dp else 0.0
            epsilons.append(epsilon)
            print(
                f"Client: {client_name} Epoch: {epoch+1} "
                f"Traffic Loss: {np.mean(losses_traffic):.4f} "
                f"Device Loss: {np.mean(losses_device):.4f} "
                f"Memory: {memory_usage:.2f} MB "
                f"CPU: {cpu_usage:.2f}% "
                f"Energy: {energy_usage:.2f} CPU-sec "
                f"| (ε = {epsilon:.2f})"
            )
        return epsilons, {
            'memory_usage_MB': np.mean(memory_usages),
            'cpu_usage_percent': np.mean(cpu_usages),
            'energy_usage_cpu_sec': np.mean(energy_usages)
        }

    def get_parameters(self):
        return list(self._model.parameters())

    def set_parameters(self, parameters):
        with torch.no_grad():
            for param, new_param in zip(self._model.parameters(), parameters):
                param.data = new_param.data.clone()

    def get_model(self):
        return self._model

class TF1Client:
    def __init__(self, X, y_traffic, y_device, input_shape, num_classes_traffic, num_classes_device, task='both'):
        self._model = TF1DCNN(input_shape, num_classes_device, num_classes_traffic, task=task)
        self.X = X
        self.y_traffic = y_traffic
        self.y_device = y_device
        self.task = task

    def train(self, epochs, client_name):
        process = psutil.Process()
        start_time = time.time()
        cpu_percent = []
        for _ in tqdm(range(epochs), desc=f'Training {client_name} TF1 Epoch'):
            if self.task == 'both':
                self._model.fit(self.X, self.y_traffic, self.y_device, epochs=1, batch_size=args.batch_size)
            elif self.task == 'traffic':
                self._model.fit(self.X, self.y_traffic, None, epochs=1, batch_size=args.batch_size)
            elif self.task == 'device':
                self._model.fit(self.X, None, self.y_device, epochs=1, batch_size=args.batch_size)
            cpu_percent.append(psutil.cpu_percent(interval=None))
        memory_usage = process.memory_info().rss / (1024 ** 2)  # MB
        cpu_usage = np.mean(cpu_percent) if cpu_percent else 0.0
        train_time = time.time() - start_time
        energy_usage = cpu_usage * train_time  # Proxy: CPU% * time (seconds)
        print(
            f"Client {client_name} trained for {epochs} epochs. "
            f"Memory: {memory_usage:.2f} MB, "
            f"CPU: {cpu_usage:.2f}%, "
            f"Energy: {energy_usage:.2f} CPU-sec"
        )
        return {
            'memory_usage_MB': memory_usage,
            'cpu_usage_percent': cpu_usage,
            'energy_usage_cpu_sec': energy_usage
        }

    def get_parameters(self):
        return self._model.get_weights()

    def set_parameters(self, weights):
        self._model.set_weights(weights)

    def get_model(self):
        return self._model

class MLClient:
    def __init__(self, model_type, X, y_traffic, y_device, multi_output=True):
        self.multi_output = multi_output
        if multi_output:
            self._model = MLModel(model_type, multi_output=True)
        else:
            base_model_type = model_type.replace('_traffic', '').replace('_device', '')
            self._model_traffic = MLModel(f"{base_model_type}_traffic", multi_output=False)
            self._model_device = MLModel(f"{base_model_type}_device", multi_output=False)
        self.X = X
        self.y_traffic = y_traffic
        self.y_device = y_device

    def train(self, epochs=None, client_name=None):
        process = psutil.Process()
        epochs = 1 if epochs is None else epochs
        start_time = time.time()
        cpu_percent = []
        for _ in tqdm(range(epochs), desc=f"Training {client_name} ML Model"):
            if self.multi_output:
                self._model.fit(self.X, self.y_traffic, self.y_device)
            else:
                self._model_traffic.fit(self.X, self.y_traffic)
                self._model_device.fit(self.X, self.y_device)
            cpu_percent.append(psutil.cpu_percent(interval=None))
        memory_usage = process.memory_info().rss / (1024 ** 2)  # MB
        cpu_usage = np.mean(cpu_percent) if cpu_percent else 0.0
        train_time = time.time() - start_time
        energy_usage = cpu_usage * train_time  # Proxy: CPU% * time (seconds)
        print(
            f"Client {client_name} trained {self._model.model_type if self.multi_output else f'{self._model_traffic.model_type}, {self._model_device.model_type}'} model(s). "
            f"Memory: {memory_usage:.2f} MB, "
            f"CPU: {cpu_usage:.2f}%, "
            f"Energy: {energy_usage:.2f} CPU-sec"
        )
        return {
            'memory_usage_MB': memory_usage,
            'cpu_usage_percent': cpu_usage,
            'energy_usage_cpu_sec': energy_usage
        }

    def get_model(self, task='both'):
        if self.multi_output:
            return self._model
        return self._model_traffic if task == 'traffic' else self._model_device

def save_ensemble_results(metrics, preds_traffic, preds_device, labels_traffic, labels_device, label_encoders, model_name, multi_output, task):
    """
    Save and print ensemble results (metrics, classification report, confusion matrix) for ML models.
    """
    # Save metrics
    os.makedirs('metrics', exist_ok=True)
    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(f'metrics/{model_name}_ensemble_final.csv', index=False)
    print(f"\nEnsemble metrics saved for {model_name} to 'metrics/{model_name}_ensemble_final.csv'")

    # Print key metrics
    print(f"\n{model_name} Ensemble Metrics:")
    table_data = [[k, f"{v:.4f}" if isinstance(v, (float, np.floating)) else v] for k, v in metrics.items()]
    print(tabulate(table_data, headers=['Metric', 'Value'], tablefmt='grid'))

    # Save and print traffic results
    if task in ['both', 'traffic']:
        traffic_metrics, traffic_report, traffic_cm = compute_metrics(
            labels_traffic, preds_traffic, 'traffic', None, label_encoders['traffic']
        )
        os.makedirs('classification_reports', exist_ok=True)
        os.makedirs('confusion_matrices', exist_ok=True)
        traffic_report.to_csv(f"classification_reports/traffic_{model_name}_ensemble_final.csv")
        traffic_cm_df = pd.DataFrame(
            traffic_cm,
            index=label_encoders['traffic'].classes_[:traffic_cm.shape[0]],
            columns=label_encoders['traffic'].classes_[:traffic_cm.shape[1]]
        )
        traffic_cm_df.to_csv(f"confusion_matrices/traffic_{model_name}_ensemble_final.csv")
        print(f"\n{model_name} Traffic Ensemble Classification Report:")
        print(traffic_report.to_string())
        print(f"\n{model_name} Traffic Ensemble Confusion Matrix:")
        print(traffic_cm_df.to_string())

    # Save and print device results
    if multi_output or task == 'device':
        device_metrics, device_report, device_cm = compute_metrics(
            labels_device, preds_device, 'device', None, label_encoders['device']
        )
        os.makedirs('classification_reports', exist_ok=True)
        os.makedirs('confusion_matrices', exist_ok=True)
        device_report.to_csv(f"classification_reports/device_{model_name}_ensemble_final.csv")
        device_cm_df = pd.DataFrame(
            device_cm,
            index=label_encoders['device'].classes_[:device_cm.shape[0]],
            columns=label_encoders['device'].classes_[:device_cm.shape[1]]
        )
        device_cm_df.to_csv(f"confusion_matrices/device_{model_name}_ensemble_final.csv")
        print(f"\n{model_name} Device Ensemble Classification Report:")
        print(device_report.to_string())
        print(f"\n{model_name} Device Ensemble Confusion Matrix:")
        print(device_cm_df.to_string())

def evaluate_dl(model, device, test_loader, round_num=None, label_encoders=None, task='both', model_name='unknown'):
    model.eval()
    model.to(device)
    process = psutil.Process()
    start_time = time.time()
    cpu_percent = []
    loss_traffic = nn.CrossEntropyLoss()
    loss_device = nn.CrossEntropyLoss()
    test_loss_traffic = 0
    test_loss_device = 0
    correct_traffic = 0
    correct_device = 0
    all_preds_traffic = []
    all_labels_traffic = []
    all_probs_traffic = []
    all_preds_device = []
    all_labels_device = []
    all_probs_device = []
    with torch.no_grad():
        for x, y in tqdm(test_loader, desc="Evaluating"):
            x = x.to(device).to(torch.float32)
            cpu_percent.append(psutil.cpu_percent(interval=None))
            if task == 'both':
                y_traffic, y_device = y[:, 0].to(device), y[:, 1].to(device)
                traffic_out, device_out = model(x)
                test_loss_traffic += loss_traffic(traffic_out, y_traffic).item()
                test_loss_device += loss_device(device_out, y_device).item()
                probs_traffic = torch.softmax(traffic_out, dim=1)
                probs_device = torch.softmax(device_out, dim=1)
                preds_traffic = torch.argmax(probs_traffic, dim=1)
                preds_device = torch.argmax(probs_device, dim=1)
                correct_traffic += preds_traffic.eq(y_traffic).sum().item()
                correct_device += preds_device.eq(y_device).sum().item()
                all_preds_traffic.extend(preds_traffic.cpu().numpy())
                all_labels_traffic.extend(y_traffic.cpu().numpy())
                all_probs_traffic.extend(probs_traffic.cpu().numpy())
                all_preds_device.extend(preds_device.cpu().numpy())
                all_labels_device.extend(y_device.cpu().numpy())
                all_probs_device.extend(probs_device.cpu().numpy())
            else:
                y_target = y[:, 0 if task == 'traffic' else 1].to(device)
                out = model(x)
                loss = loss_traffic(out, y_target) if task == 'traffic' else loss_device(out, y_target)
                probs = torch.softmax(out, dim=1)
                preds = torch.argmax(probs, dim=1)
                correct = preds.eq(y_target).sum().item()
                if task == 'traffic':
                    test_loss_traffic += loss.item()
                    correct_traffic += correct
                    all_preds_traffic.extend(preds.cpu().numpy())
                    all_labels_traffic.extend(y_target.cpu().numpy())
                    all_probs_traffic.extend(probs.cpu().numpy())
                else:
                    test_loss_device += loss.item()
                    correct_device += correct
                    all_preds_device.extend(preds.cpu().numpy())
                    all_labels_device.extend(y_target.cpu().numpy())
                    all_probs_device.extend(probs.cpu().numpy())
    memory_usage = process.memory_info().rss / (1024 ** 2)  # MB
    cpu_usage = np.mean(cpu_percent) if cpu_percent else 0.0
    test_time = time.time() - start_time
    energy_usage = cpu_usage * test_time  # Proxy: CPU% * time (seconds)
    test_loss_traffic /= len(test_loader.dataset)
    test_loss_device /= len(test_loader.dataset)
    accuracy_traffic = correct_traffic / len(test_loader.dataset) if task in ['both', 'traffic'] else 0
    accuracy_device = correct_device / len(test_loader.dataset) if task in ['both', 'device'] else 0
    metrics = {}
    if task in ['both', 'traffic']:
        traffic_metrics, traffic_report, traffic_cm = compute_metrics(
            np.array(all_labels_traffic), np.array(all_preds_traffic), 'traffic', np.array(all_probs_traffic), label_encoders['traffic']
        )
        metrics.update(traffic_metrics)
        report_path = f"classification_reports/traffic_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/traffic_{model_name}_final.csv"
        traffic_report.to_csv(report_path)
        traffic_cm_df = pd.DataFrame(
            traffic_cm,
            index=label_encoders['traffic'].classes_[:traffic_cm.shape[0]],
            columns=label_encoders['traffic'].classes_[:traffic_cm.shape[0]]
        )
        traffic_cm_df.to_csv(f"confusion_matrices/traffic_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/traffic_{model_name}_final.csv")
    if task in ['both', 'device']:
        device_metrics, device_report, device_cm = compute_metrics(
            np.array(all_labels_device), np.array(all_preds_device), 'device', np.array(all_probs_device), label_encoders['device']
        )
        metrics.update(device_metrics)
        report_path = f"classification_reports/device_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/device_{model_name}_final.csv"
        device_cm_df = pd.DataFrame(
            device_cm,
            index=label_encoders['device'].classes_[:device_cm.shape[0]],
            columns=label_encoders['device'].classes_[:device_cm.shape[0]]
        )
        device_cm_df.to_csv(f"confusion_matrices/device_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/device_{model_name}_final.csv")
    metrics.update({
        'traffic_accuracy': accuracy_traffic,
        'device_accuracy': accuracy_device,
        'traffic_loss': test_loss_traffic,
        'device_loss': test_loss_device,
        'test_memory_usage_MB': memory_usage,
        'test_cpu_usage_percent': cpu_usage,
        'test_energy_usage_cpu_sec': energy_usage
    })
    return metrics, (all_preds_traffic, all_preds_device), (all_labels_traffic, all_labels_device), (all_probs_traffic, all_probs_device)


def evaluate_tf1(model, X_test, y_traffic_test, y_device_test, round_num=None, label_encoders=None, task='both', model_name='unknown'):
    process = psutil.Process()
    start_time = time.time()
    cpu_percent = []
    attack_preds, device_preds, attack_probs, device_probs = model.predict(X_test)
    cpu_percent.append(psutil.cpu_percent(interval=None))
    memory_usage = process.memory_info().rss / (1024 ** 2)  # MB
    cpu_usage = np.mean(cpu_percent) if cpu_percent else 0.0
    test_time = time.time() - start_time
    energy_usage = cpu_usage * test_time  # Proxy: CPU% * time (seconds)
    metrics = {}
    if task in ['both', 'traffic']:
        traffic_metrics, traffic_report, traffic_cm = compute_metrics(
            y_traffic_test, attack_preds, 'traffic', attack_probs, label_encoders['traffic']
        )
        metrics.update(traffic_metrics)
        report_path = f"classification_reports/traffic_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/traffic_{model_name}_final.csv"
        traffic_report.to_csv(report_path)
        traffic_cm_df = pd.DataFrame(
            traffic_cm,
            index=label_encoders['traffic'].classes_[:traffic_cm.shape[0]],
            columns=label_encoders['traffic'].classes_[:traffic_cm.shape[0]]
        )
        traffic_cm_df.to_csv(f"confusion_matrices/traffic_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/traffic_{model_name}_final.csv")
    if task in ['both', 'device']:
        device_metrics, device_report, device_cm = compute_metrics(
            y_device_test, device_preds, 'device', device_probs, label_encoders['device']
        )
        metrics.update(device_metrics)
        report_path = f"classification_reports/device_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/device_{model_name}_final.csv"
        device_cm_df = pd.DataFrame(
            device_cm,
            index=label_encoders['device'].classes_[:device_cm.shape[0]],
            columns=label_encoders['device'].classes_[:device_cm.shape[0]]
        )
        device_cm_df.to_csv(f"confusion_matrices/device_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/device_{model_name}_final.csv")
    metrics.update({
        'traffic_accuracy': np.mean(attack_preds == y_traffic_test) if task in ['both', 'traffic'] else 0.0,
        'device_accuracy': np.mean(device_preds == y_device_test) if task in ['both', 'device'] else 0.0,
        'traffic_loss': 0.0,
        'device_loss': 0.0,
        'test_memory_usage_MB': memory_usage,
        'test_cpu_usage_percent': cpu_usage,
        'test_energy_usage_cpu_sec': energy_usage
    })
    return metrics, (attack_preds, device_preds), (y_traffic_test, y_device_test), (attack_probs, device_probs)


def evaluate_ml(model, X_test, y_traffic_test, y_device_test, round_num=None, label_encoders=None, multi_output=True, task='both', model_name='unknown'):
    process = psutil.Process()
    start_time = time.time()
    cpu_percent = []
    attack_preds, device_preds, attack_probs, device_probs = model.predict(X_test)
    cpu_percent.append(psutil.cpu_percent(interval=None))
    memory_usage = process.memory_info().rss / (1024 ** 2)  # MB
    cpu_usage = np.mean(cpu_percent) if cpu_percent else 0.0
    test_time = time.time() - start_time
    energy_usage = cpu_usage * test_time  # Proxy: CPU% * time (seconds)
    metrics = {}
    if multi_output:
        if task in ['both', 'traffic']:
            traffic_metrics, traffic_report, traffic_cm = compute_metrics(
                y_traffic_test, attack_preds, 'traffic', attack_probs, label_encoders['traffic']
            )
            metrics.update(traffic_metrics)
            report_path = f"classification_reports/traffic_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/traffic_{model_name}_final.csv"
            traffic_report.to_csv(report_path)
            traffic_cm_df = pd.DataFrame(
                traffic_cm,
                index=label_encoders['traffic'].classes_[:traffic_cm.shape[0]],
                columns=label_encoders['traffic'].classes_[:traffic_cm.shape[0]]
            )
            traffic_cm_df.to_csv(f"confusion_matrices/traffic_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/traffic_{model_name}_final.csv")
        if task in ['both', 'device']:
            device_metrics, device_report, device_cm = compute_metrics(
                y_device_test, device_preds, 'device', device_probs, label_encoders['device']
            )
            metrics.update(device_metrics)
            report_path = f"classification_reports/device_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/device_{model_name}_final.csv"
            device_cm_df = pd.DataFrame(
                device_cm,
                index=label_encoders['device'].classes_[:device_cm.shape[0]],
                columns=label_encoders['device'].classes_[:device_cm.shape[0]]
            )
            device_cm_df.to_csv(f"confusion_matrices/device_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/device_{model_name}_final.csv")
        metrics['traffic_accuracy'] = np.mean(attack_preds == y_traffic_test) if task in ['both', 'traffic'] else 0.0
        metrics['device_accuracy'] = np.mean(device_preds == y_device_test) if task in ['both', 'device'] else 0.0
    else:
        target = 'traffic' if model.model_type.endswith('_traffic') else 'device'
        target_labels = y_traffic_test if target == 'traffic' else y_device_test
        target_probs = attack_probs
        target_encoder = label_encoders[target]
        target_metrics, target_report, target_cm = compute_metrics(
            target_labels, attack_preds, target, target_probs, target_encoder
        )
        metrics.update(target_metrics)
        report_path = f"classification_reports/{target}_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/{target}_{model_name}_final.csv"
        target_report.to_csv(report_path)
        target_cm_df = pd.DataFrame(
            target_cm,
            index=target_encoder.classes_[:target_cm.shape[0]],
            columns=target_encoder.classes_[:target_cm.shape[0]]
        )
        target_cm_df.to_csv(f"confusion_matrices/{target}_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/{target}_{model_name}_final.csv")
        metrics[f'{target}_accuracy'] = np.mean(attack_preds == target_labels)
        metrics['traffic_accuracy'] = metrics.get('traffic_accuracy', 0.0) if target == 'traffic' else 0.0
        metrics['device_accuracy'] = metrics.get('device_accuracy', 0.0) if target == 'device' else 0.0
    metrics.update({
        'traffic_loss': 0.0,
        'device_loss': 0.0,
        'test_memory_usage_MB': memory_usage,
        'test_cpu_usage_percent': cpu_usage,
        'test_energy_usage_cpu_sec': energy_usage
    })
    return metrics, (attack_preds, device_preds), (y_traffic_test, y_device_test), (attack_probs, device_probs)


def save_pytorch_model(model, scaler, label_encoders, feature_columns, filepath, task='both'):
    if not args.multi_output and task == 'both':
        # Save two separate models for single-output mode
        torch.save({
            'model_state_dict': model.state_dict(),
            'scaler': scaler,
            'label_encoder_traffic_classes': label_encoders['traffic'].classes_,
            'label_encoder_device_classes': [],
            'feature_columns': feature_columns,
            'task': 'traffic'
        }, filepath.replace('.pth', '_traffic.pth'))
        torch.save({
            'model_state_dict': model.state_dict(),
            'scaler': scaler,
            'label_encoder_traffic_classes': [],
            'label_encoder_device_classes': label_encoders['device'].classes_,
            'feature_columns': feature_columns,
            'task': 'device'
        }, filepath.replace('.pth', '_device.pth'))
    else:
        # Save single model for multi-output or specific task
        torch.save({
            'model_state_dict': model.state_dict(),
            'scaler': scaler,
            'label_encoder_traffic_classes': label_encoders['traffic'].classes_ if task in ['both', 'traffic'] else [],
            'label_encoder_device_classes': label_encoders['device'].classes_ if task in ['both', 'device'] else [],
            'feature_columns': feature_columns,
            'task': task
        }, filepath)

def save_tf_model(model, filepath):
    if not args.multi_output and model.task == 'both':
        # Save two separate models for single-output mode
        model.model.save(filepath.replace('.h5', '_traffic.h5'))
        model.model.save(filepath.replace('.h5', '_device.h5'))
    else:
        # Save single model for multi-output or specific task
        model.model.save(filepath)

def save_privacy_budget(privacy_log, filepath='privacy_budget.csv'):
    privacy_df = pd.DataFrame(privacy_log)
    privacy_df.to_csv(filepath, index=False)
    print(f"\nPrivacy budget saved to '{filepath}'")

def save_ml_model(model, scaler, label_encoders, feature_columns, filepath, multi_output=True):
    joblib.dump({
        'model': model,
        'scaler': scaler,
        'label_encoders': label_encoders,
        'feature_columns': feature_columns,
        'multi_output': multi_output
    }, filepath)

def plot_confusion_matrix(labels, predictions, savefile, label_encoder, label_type):
    cm = confusion_matrix(labels, predictions)
    class_names = label_encoder.classes_
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix ({label_type})")
    plt.savefig(savefile)
    plt.close()


# if __name__ == "__main__":
#     csv_file = r"F:/Datasets_Project/NIM LAB IoT Dataset 2025/slowloriss/small/processed/NIMLABIoT_processed.csv"
#     X_scaled, y_device_encoded, y_attack_encoded, le_device, le_attack, feature_columns = load_and_preprocess_data(csv_file)
#     print(f"\nNumber of features used: {len(feature_columns)}")
#     print("Features used:", feature_columns)
#     dataloader = DatasetLoader(X_scaled, y_device_encoded, y_attack_encoded, le_device, le_attack,
#                               num_clients=args.num_clients, sample_rate=args.sample_rate)
#     dataloader.print_client_label_distribution()
#     train_loader = dataloader.get_train_data()
#     test_loader = dataloader.get_test_data()
#     X_train, y_train_traffic, y_train_device, X_test, y_test_traffic, y_test_device = dataloader.get_ml_data()
#     input_shape = X_scaled.shape[1]
#     num_classes_traffic = len(np.unique(y_attack_encoded))
#     num_classes_device = len(np.unique(y_device_encoded))
#     clients_data = dataloader.clients_data
#     local_epochs = max(1, args.epochs // args.communication_rounds)

#     models = []
#     if args.model_type in ['dl', 'both']:
#         if args.multi_output:
#             models.extend([
#                 {'name': 'cnn', 'type': 'dl', 'class': MultiOutputCNN, 'server': DLServer, 'task': 'both'},
#                 {'name': 'lstm', 'type': 'dl', 'class': LSTMModel, 'server': DLServer, 'task': 'both'},
#                 {'name': 'bilstm', 'type': 'dl', 'class': BiLSTM, 'server': DLServer, 'task': 'both'},
#                 {'name': 'tf1cnn', 'type': 'tf', 'class': TF1DCNN, 'server': TF1Server, 'task': 'both'}
#             ])
#         else:
#             models.extend([
#                 {'name': 'cnn_traffic', 'type': 'dl', 'class': MultiOutputCNN, 'server': DLServer, 'task': 'traffic'},
#                 {'name': 'cnn_device', 'type': 'dl', 'class': MultiOutputCNN, 'server': DLServer, 'task': 'device'},
#                 {'name': 'lstm_traffic', 'type': 'dl', 'class': LSTMModel, 'server': DLServer, 'task': 'traffic'},
#                 {'name': 'lstm_device', 'type': 'dl', 'class': LSTMModel, 'server': DLServer, 'task': 'device'},
#                 {'name': 'bilstm_traffic', 'type': 'dl', 'class': BiLSTM, 'server': DLServer, 'task': 'traffic'},
#                 {'name': 'bilstm_device', 'type': 'dl', 'class': BiLSTM, 'server': DLServer, 'task': 'device'},
#                 {'name': 'tf1cnn_traffic', 'type': 'tf', 'class': TF1DCNN, 'server': TF1Server, 'task': 'traffic'},
#                 {'name': 'tf1cnn_device', 'type': 'tf', 'class': TF1DCNN, 'server': TF1Server, 'task': 'device'}
#             ])
#     if args.model_type in ['ml', 'both']:
#         if args.multi_output:
#             models.extend([
#                 {'name': 'xgboost', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'xgboost', 'multi_output': True}},
#                 {'name': 'rf', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'rf', 'multi_output': True}},
#                 {'name': 'dt', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'dt', 'multi_output': True}}
#             ])
#         else:
#             models.extend([
#                 {'name': 'xgboost_traffic', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'xgboost_traffic', 'multi_output': False}},
#                 {'name': 'xgboost_device', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'xgboost_device', 'multi_output': False}},
#                 {'name': 'rf_traffic', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'rf_traffic', 'multi_output': False}},
#                 {'name': 'rf_device', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'rf_device', 'multi_output': False}},
#                 {'name': 'dt_traffic', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'dt_traffic', 'multi_output': False}},
#                 {'name': 'dt_device', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'dt_device', 'multi_output': False}}
#             ])

#     metrics_log = []
#     privacy_log = []

#     for model_config in models:
#         model_name = model_config['name']
#         model_type = model_config['type']
#         task = model_config.get('task', 'both')
#         print(f"\nTraining model: {model_name}")
#         model_train_start_time = time.time()

#         if model_type == 'dl':
#             server = model_config['server'](
#                 model_class=model_config['class'],
#                 input_size=input_shape,
#                 num_classes_traffic=num_classes_traffic,
#                 num_classes_device=num_classes_device,
#                 task=task
#             )
#             clients = {
#                 client_name: DLClient(
#                     model_class=model_config['class'],
#                     data_loader=client_data['dl_loader'],
#                     input_size=input_shape,
#                     num_classes_traffic=num_classes_traffic,
#                     num_classes_device=num_classes_device,
#                     y_train_traffic=client_data['y_traffic'],
#                     y_train_device=client_data['y_device'],
#                     task=task
#                 )
#                 for client_name, client_data in clients_data.items()
#             }
#         elif model_type == 'tf':
#             server = model_config['server'](
#                 input_shape=input_shape,
#                 num_classes_traffic=num_classes_traffic,
#                 num_classes_device=num_classes_device,
#                 task=task
#             )
#             clients = {
#                 client_name: TF1Client(
#                     X=client_data['X'],
#                     y_traffic=client_data['y_traffic'],
#                     y_device=client_data['y_device'],
#                     input_shape=input_shape,
#                     num_classes_traffic=num_classes_traffic,
#                     num_classes_device=num_classes_device,
#                     task=task
#                 )
#                 for client_name, client_data in clients_data.items()
#             }
#         elif model_type == 'ml':
#             server = model_config['server'](**model_config['params'])
#             clients = {
#                 client_name: MLClient(
#                     model_type=model_config['params']['model_type'],
#                     X=client_data['X'],
#                     y_traffic=client_data['y_traffic'],
#                     y_device=client_data['y_device'],
#                     multi_output=model_config['params']['multi_output']
#                 )
#                 for client_name, client_data in clients_data.items()
#             }

#         client_metrics = {}  # Store per-client training metrics
#         for comm_round in range(args.communication_rounds):
#             print(f"\n{model_name} Communication Round {comm_round + 1}/{args.communication_rounds}")
#             round_train_start_time = time.time()
#             round_train_metrics = {
#                 'memory_usage_MB': [],
#                 'cpu_usage_percent': [],
#                 'energy_usage_cpu_sec': []
#             }

#             for client_name, client in clients.items():
#                 print(f"\nTraining {client_name}")
#                 if model_type == 'dl':
#                     client.set_parameters(server.get_model().parameters())
#                     epsilons, train_metrics = client.train(local_epochs, client_name)
#                     server.add_client(client.get_parameters(), len(client.data_loader.dataset))
#                     for epoch, epsilon in enumerate(epsilons, 1):
#                         privacy_log.append({
#                             'model': model_name,
#                             'client': client_name,
#                             'communication_round': comm_round + 1,
#                             'epoch': epoch,
#                             'epsilon': epsilon
#                         })
#                 elif model_type == 'tf':
#                     client.set_parameters(server.get_model().get_weights())
#                     train_metrics = client.train(local_epochs, client_name)
#                     server.add_client(client.get_parameters(), len(client.X))
#                 elif model_type == 'ml':
#                     train_metrics = client.train(local_epochs, client_name)
#                     if model_config['params']['multi_output']:
#                         server.add_client(client.get_model(), client_name, client.X, client.y_traffic, client.y_device)
#                     else:
#                         if task == 'traffic' or model_name.endswith('_traffic'):
#                             server.add_client(client.get_model(task='traffic'), client_name, client.X, client.y_traffic, None)
#                         if task == 'device' or model_name.endswith('_device'):
#                             server.add_client(client.get_model(task='device'), client_name, client.X, None, client.y_device)
#                 client_metrics[client_name] = client_metrics.get(client_name, []) + [train_metrics]
#                 for key in round_train_metrics:
#                     round_train_metrics[key].append(train_metrics[key])

#             # Aggregate and evaluate
#             test_start_time = time.time()
#             if model_type == 'ml' and task == 'both':
#                 attack_preds, device_preds, attack_probs, device_probs = server.aggregate(X_test)
#                 metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_ml(
#                     server.get_model(),
#                     X_test,
#                     y_test_traffic,
#                     y_test_traffic,
#                     comm_round + 1,
#                     dataloader.label_encoders,
#                     multi_output=model_config['params']['multi_output'],
#                     task=task,
#                     model_name=model_name
#                 )
#             elif model_type == 'ml':
#                 attack_preds, device_preds, attack_probs, device_probs = server.aggregate(X_test)
#                 metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_ml(
#                     server.get_model(),
#                     X_test,
#                     y_test_traffic,
#                     y_test_device,
#                     comm_round + 1,
#                     dataloader.label_encoders,
#                     multi_output=model_config['params']['multi_output'],
#                     task=task,
#                     model_name=model_name
#                 )
#             elif model_type == 'dl':
#                 server.aggregate()
#                 metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_dl(
#                     server.get_model(),
#                     args.device,
#                     test_loader,
#                     comm_round + 1,
#                     dataloader.label_encoders,
#                     task=task,
#                     model_name=model_name
#                 )
#             elif model_type == 'tf':
#                 server.aggregate()
#                 metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_tf1(
#                     server.get_model(),
#                     X_test,
#                     y_test_traffic,
#                     y_test_device,
#                     comm_round + 1,
#                     dataloader.label_encoders,
#                     task=task,
#                     model_name=model_name
#                 )

#             test_end_time = time.time()
#             test_time = test_end_time - test_start_time
#             round_train_time = time.time() - round_train_start_time - test_time
#             round_metrics = metrics.copy()
#             round_metrics.update({
#                 'model': model_name,
#                 'round': comm_round + 1,
#                 'train_time': round_train_time,
#                 'test_time': test_time,
#                 'test_per_sample': test_time / len(X_test),
#                 'train_memory_usage_MB': np.mean(round_train_metrics['memory_usage_MB']),
#                 'train_cpu_usage_percent': np.mean(round_train_metrics['cpu_usage_percent']),
#                 'train_energy_usage_cpu_sec': np.mean(round_train_metrics['energy_usage_cpu_sec'])
#             })
#             metrics_log.append(round_metrics)
#             print(f"\n{model_name} Metrics after Round {comm_round + 1}:")
#             table_data = [[k, f"{v:.4f}" if isinstance(v, (float, np.floating)) else v] for k, v in round_metrics.items()]
#             print(tabulate(table_data, headers=['Metric', 'Value'], tablefmt='grid'))

#         # Calculate total training time for the model
#         model_train_time = time.time() - model_train_start_time

#         # For ML models, generate and save ensemble results
#         if model_type == 'ml':
#             print(f"\nGenerating ensemble results for {model_name}")
#             metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_ml(
#                 server.get_model(),
#                 X_test,
#                 y_test_traffic,
#                 y_test_device,
#                 None,  # Final evaluation
#                 dataloader.label_encoders,
#                 multi_output=model_config['params']['multi_output'],
#                 task=task,
#                 model_name=model_name
#             )
#             save_ensemble_results(
#                 metrics,
#                 preds_traffic,
#                 preds_device,
#                 labels_traffic,
#                 labels_device,
#                 dataloader.label_encoders,
#                 model_name,
#                 model_config['params']['multi_output'],
#                 task
#             )
#             # Update metrics with final model training time and average client metrics
#             client_avg_metrics = {
#                 'train_memory_usage_MB': np.mean([m['memory_usage_MB'] for client in client_metrics.values() for m in client]),
#                 'train_cpu_usage_percent': np.mean([m['cpu_usage_percent'] for client in client_metrics.values() for m in client]),
#                 'train_energy_usage_cpu_sec': np.mean([m['energy_usage_cpu_sec'] for client in client_metrics.values() for m in client])
#             }
#             final_metrics = metrics.copy()
#             final_metrics.update({
#                 'model': model_name,
#                 'round': 'ensemble',
#                 'train_time': model_train_time,
#                 'test_time': test_time,
#                 'test_per_sample': test_time / len(X_test),
#                 'train_memory_usage_MB': client_avg_metrics['train_memory_usage_MB'],
#                 'train_cpu_usage_percent': client_avg_metrics['train_cpu_usage_percent'],
#                 'train_energy_usage_cpu_sec': client_avg_metrics['train_energy_usage_cpu_sec']
#             })
#             metrics_log.append(final_metrics)

#         # Print privacy budgets for DL models
#         if model_type == 'dl':
#             print(f"\nPrivacy Budgets for {model_name}:")
#             for entry in privacy_log:
#                 if entry['model'] == model_name:
#                     print(
#                         f"Client: {entry['client']}, "
#                         f"Round: {entry['communication_round']}, "
#                         f"Epoch: {entry['epoch']}, "
#                         f"Epsilon: {entry['epsilon']:.2f}"
#                     )

#         # Save final model
#         os.makedirs('models', exist_ok=True)
#         os.makedirs('confusion_matrices', exist_ok=True)
#         if model_type == 'dl':
#             save_pytorch_model(
#                 server.get_model(),
#                 dataloader.scaler,
#                 dataloader.label_encoders,
#                 feature_columns,
#                 f'models/{model_name}_final.pth',
#                 task=task
#             )
#             if args.multi_output or task == 'traffic':
#                 plot_confusion_matrix(labels_traffic, preds_traffic, f'confusion_matrices/{model_name}_traffic_final.png', dataloader.label_encoders['traffic'], 'Traffic Type')
#             if args.multi_output or task == 'device':
#                 plot_confusion_matrix(labels_device, preds_device, f'confusion_matrices/{model_name}_device_final.png', dataloader.label_encoders['device'], 'Device')
#         elif model_type == 'tf':
#             save_tf_model(server.get_model(), f'models/{model_name}_final.h5')
#             if args.multi_output or task == 'traffic':
#                 plot_confusion_matrix(labels_traffic, preds_traffic, f'confusion_matrices/{model_name}_traffic_final.png', dataloader.label_encoders['traffic'], 'Traffic Type')
#             if args.multi_output or task == 'device':
#                 plot_confusion_matrix(labels_device, preds_device, f'confusion_matrices/{model_name}_device_final.png', dataloader.label_encoders['device'], 'Device')
#         elif model_type == 'ml':
#             for client_name, client in clients.items():
#                 if model_config['params']['multi_output']:
#                     save_ml_model(
#                         client.get_model(),
#                         dataloader.scaler,
#                         dataloader.label_encoders,
#                         feature_columns,
#                         f'models/{model_name}_{client_name}_final.joblib',
#                         multi_output=True
#                     )
#                 else:
#                     if task == 'traffic' or model_name.endswith('_traffic'):
#                         save_ml_model(
#                             client.get_model(task='traffic'),
#                             dataloader.scaler,
#                             dataloader.label_encoders,
#                             feature_columns,
#                             f'models/{model_name}_{client_name}_traffic_final.joblib',
#                             multi_output=False
#                         )
#                     if task == 'device' or model_name.endswith('_device'):
#                         save_ml_model(
#                             client.get_model(task='device'),
#                             dataloader.scaler,
#                             dataloader.label_encoders,
#                             feature_columns,
#                             f'models/{model_name}_{client_name}_device_final.joblib',
#                             multi_output=False
#                         )
#             if args.multi_output or task == 'traffic':
#                 plot_confusion_matrix(labels_traffic, preds_traffic, f'confusion_matrices/{model_name}_traffic_final.png', dataloader.label_encoders['traffic'], 'Traffic Type')
#             if (args.multi_output or task == 'device') and preds_device is not None:
#                 plot_confusion_matrix(labels_device, preds_device, f'confusion_matrices/{model_name}_device_final.png', dataloader.label_encoders['device'], 'Device')

#     # Save metrics to CSV
#     os.makedirs('metrics', exist_ok=True)
#     metrics_df = pd.DataFrame(metrics_log)
#     metrics_df.to_csv('metrics_final.csv', index=False)
#     print("\nFinal metrics saved to 'metrics_final.csv'")

#     # Save privacy budget to CSV
#     if privacy_log:
#         save_privacy_budget(privacy_log)

No GPU detected. Running TensorFlow on CPU.
TensorFlow version: 2.19.0


In [2]:
if __name__ == "__main__":
    csv_file = r"F:/Datasets_Project/NIM LAB IoT Dataset 2025/slowloriss/small/processed/NIMLABIoT_processed.csv"
    X_scaled, y_device_encoded, y_attack_encoded, le_device, le_attack, feature_columns = load_and_preprocess_data(csv_file)
    print(f"\nNumber of features used: {len(feature_columns)}")
    print("Features used:", feature_columns)
    dataloader = DatasetLoader(X_scaled, y_device_encoded, y_attack_encoded, le_device, le_attack,
                              num_clients=args.num_clients, sample_rate=args.sample_rate)
    dataloader.print_client_label_distribution()
    train_loader = dataloader.get_train_data()
    test_loader = dataloader.get_test_data()
    X_train, y_train_traffic, y_train_device, X_test, y_test_traffic, y_test_device = dataloader.get_ml_data()
    input_shape = X_scaled.shape[1]
    num_classes_traffic = len(np.unique(y_attack_encoded))
    num_classes_device = len(np.unique(y_device_encoded))
    clients_data = dataloader.clients_data
    local_epochs = max(1, args.epochs // args.communication_rounds)

    models = []
    if args.model_type in ['dl', 'both']:
        if args.multi_output:
            models.extend([
                {'name': 'cnn', 'type': 'dl', 'class': MultiOutputCNN, 'server': DLServer, 'task': 'both'},
                {'name': 'lstm', 'type': 'dl', 'class': LSTMModel, 'server': DLServer, 'task': 'both'},
                {'name': 'bilstm', 'type': 'dl', 'class': BiLSTM, 'server': DLServer, 'task': 'both'},
                {'name': 'tf1cnn', 'type': 'tf', 'class': TF1DCNN, 'server': TF1Server, 'task': 'both'}
            ])
        else:
            models.extend([
                {'name': 'cnn_traffic', 'type': 'dl', 'class': MultiOutputCNN, 'server': DLServer, 'task': 'traffic'},
                {'name': 'cnn_device', 'type': 'dl', 'class': MultiOutputCNN, 'server': DLServer, 'task': 'device'},
                {'name': 'lstm_traffic', 'type': 'dl', 'class': LSTMModel, 'server': DLServer, 'task': 'traffic'},
                {'name': 'lstm_device', 'type': 'dl', 'class': LSTMModel, 'server': DLServer, 'task': 'device'},
                {'name': 'bilstm_traffic', 'type': 'dl', 'class': BiLSTM, 'server': DLServer, 'task': 'traffic'},
                {'name': 'bilstm_device', 'type': 'dl', 'class': BiLSTM, 'server': DLServer, 'task': 'device'},
                {'name': 'tf1cnn_traffic', 'type': 'tf', 'class': TF1DCNN, 'server': TF1Server, 'task': 'traffic'},
                {'name': 'tf1cnn_device', 'type': 'tf', 'class': TF1DCNN, 'server': TF1Server, 'task': 'device'}
            ])
    if args.model_type in ['ml', 'both']:
        if args.multi_output:
            models.extend([
                {'name': 'xgboost', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'xgboost', 'multi_output': True}},
                {'name': 'rf', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'rf', 'multi_output': True}},
                {'name': 'dt', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'dt', 'multi_output': True}}
            ])
        else:
            models.extend([
                {'name': 'xgboost_traffic', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'xgboost_traffic', 'multi_output': False}},
                {'name': 'xgboost_device', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'xgboost_device', 'multi_output': False}},
                {'name': 'rf_traffic', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'rf_traffic', 'multi_output': False}},
                {'name': 'rf_device', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'rf_device', 'multi_output': False}},
                {'name': 'dt_traffic', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'dt_traffic', 'multi_output': False}},
                {'name': 'dt_device', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'dt_device', 'multi_output': False}}
            ])

    metrics_log = []
    privacy_log = []

    for model_config in models:
        model_name = model_config['name']
        model_type = model_config['type']
        task = model_config.get('task', 'both')
        print(f"\nTraining model: {model_name}")
        model_train_start_time = time.time()

        if model_type == 'dl':
            server = model_config['server'](
                model_class=model_config['class'],
                input_size=input_shape,
                num_classes_traffic=num_classes_traffic,
                num_classes_device=num_classes_device,
                task=task
            )
            clients = {
                client_name: DLClient(
                    model_class=model_config['class'],
                    data_loader=client_data['dl_loader'],
                    input_size=input_shape,
                    num_classes_traffic=num_classes_traffic,
                    num_classes_device=num_classes_device,
                    y_train_traffic=client_data['y_traffic'],
                    y_train_device=client_data['y_device'],
                    task=task
                )
                for client_name, client_data in clients_data.items()
            }
        elif model_type == 'tf':
            server = model_config['server'](
                input_shape=input_shape,
                num_classes_traffic=num_classes_traffic,
                num_classes_device=num_classes_device,
                task=task
            )
            clients = {
                client_name: TF1Client(
                    X=client_data['X'],
                    y_traffic=client_data['y_traffic'],
                    y_device=client_data['y_device'],
                    input_shape=input_shape,
                    num_classes_traffic=num_classes_traffic,
                    num_classes_device=num_classes_device,
                    task=task
                )
                for client_name, client_data in clients_data.items()
            }
        elif model_type == 'ml':
            server = model_config['server'](**model_config['params'])
            clients = {
                client_name: MLClient(
                    model_type=model_config['params']['model_type'],
                    X=client_data['X'],
                    y_traffic=client_data['y_traffic'],
                    y_device=client_data['y_device'],
                    multi_output=model_config['params']['multi_output']
                )
                for client_name, client_data in clients_data.items()
            }

        client_metrics = {}  # Store per-client training metrics
        for comm_round in range(args.communication_rounds):
            print(f"\n{model_name} Communication Round {comm_round + 1}/{args.communication_rounds}")
            round_train_start_time = time.time()
            round_train_metrics = {
                'memory_usage_MB': [],
                'cpu_usage_percent': [],
                'energy_usage_cpu_sec': []
            }

            for client_name, client in clients.items():
                print(f"\nTraining {client_name}")
                if model_type == 'dl':
                    client.set_parameters(server.get_model().parameters())
                    epsilons, train_metrics = client.train(local_epochs, client_name)
                    server.add_client(client.get_parameters(), len(client.data_loader.dataset))
                    for epoch, epsilon in enumerate(epsilons, 1):
                        privacy_log.append({
                            'model': model_name,
                            'client': client_name,
                            'communication_round': comm_round + 1,
                            'epoch': epoch,
                            'epsilon': epsilon
                        })
                elif model_type == 'tf':
                    client.set_parameters(server.get_model().get_weights())
                    train_metrics = client.train(local_epochs, client_name)
                    server.add_client(client.get_parameters(), len(client.X))
                elif model_type == 'ml':
                    train_metrics = client.train(local_epochs, client_name)
                    if model_config['params']['multi_output']:
                        server.add_client(client.get_model(), client_name, client.X, client.y_traffic, client.y_device)
                    else:
                        if task == 'traffic' or model_name.endswith('_traffic'):
                            server.add_client(client.get_model(task='traffic'), client_name, client.X, client.y_traffic, None)
                        if task == 'device' or model_name.endswith('_device'):
                            server.add_client(client.get_model(task='device'), client_name, client.X, None, client.y_device)
                client_metrics[client_name] = client_metrics.get(client_name, []) + [train_metrics]
                for key in round_train_metrics:
                    round_train_metrics[key].append(train_metrics[key])

            # Aggregate and evaluate
            test_start_time = time.time()
            if model_type == 'ml' and task == 'both':
                attack_preds, device_preds, attack_probs, device_probs = server.aggregate(X_test)
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_ml(
                    server.get_model(),
                    X_test,
                    y_test_traffic,
                    y_test_device,  # Fixed typo from previous version
                    comm_round + 1,
                    dataloader.label_encoders,
                    multi_output=model_config['params']['multi_output'],
                    task=task,
                    model_name=model_name
                )
            elif model_type == 'ml':
                attack_preds, device_preds, attack_probs, device_probs = server.aggregate(X_test)
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_ml(
                    server.get_model(),
                    X_test,
                    y_test_traffic,
                    y_test_device,
                    comm_round + 1,
                    dataloader.label_encoders,
                    multi_output=model_config['params']['multi_output'],
                    task=task,
                    model_name=model_name
                )
            elif model_type == 'dl':
                server.aggregate()
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_dl(
                    server.get_model(),
                    args.device,
                    test_loader,
                    comm_round + 1,
                    dataloader.label_encoders,
                    task=task,
                    model_name=model_name
                )
            elif model_type == 'tf':
                server.aggregate()
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_tf1(
                    server.get_model(),
                    X_test,
                    y_test_traffic,
                    y_test_device,
                    comm_round + 1,
                    dataloader.label_encoders,
                    task=task,
                    model_name=model_name
                )

            test_end_time = time.time()
            test_time = test_end_time - test_start_time
            round_train_time = time.time() - round_train_start_time - test_time
            round_metrics = metrics.copy()
            round_metrics.update({
                'model': model_name,
                'round': comm_round + 1,
                'train_time': round_train_time,
                'test_time': test_time,
                'test_per_sample': test_time / len(X_test),
                'train_memory_usage_MB': np.mean(round_train_metrics['memory_usage_MB']),
                'train_cpu_usage_percent': np.mean(round_train_metrics['cpu_usage_percent']),
                'train_energy_usage_cpu_sec': np.mean(round_train_metrics['energy_usage_cpu_sec'])
            })
            metrics_log.append(round_metrics)
            print(f"\n{model_name} Metrics after Round {comm_round + 1}:")
            table_data = [[k, f"{v:.4f}" if isinstance(v, (float, np.floating)) else v] for k, v in round_metrics.items()]
            print(tabulate(table_data, headers=['Metric', 'Value'], tablefmt='grid'))

        # Calculate total training time for the model
        model_train_time = time.time() - model_train_start_time

        # Final evaluation of the aggregated server model
        print(f"\nGenerating final aggregated results for {model_name}")
        try:
            if model_type == 'dl':
                server.aggregate()  # Ensure final aggregation
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_dl(
                    server.get_model(),
                    args.device,
                    test_loader,
                    None,  # Final evaluation
                    dataloader.label_encoders,
                    task=task,
                    model_name=model_name
                )
                save_ensemble_results(
                    metrics,
                    preds_traffic,
                    preds_device,
                    labels_traffic,
                    labels_device,
                    dataloader.label_encoders,
                    model_name,
                    args.multi_output,
                    task
                )
            elif model_type == 'tf':
                server.aggregate()  # Ensure final aggregation
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_tf1(
                    server.get_model(),
                    X_test,
                    y_test_traffic,
                    y_test_device,
                    None,  # Final evaluation
                    dataloader.label_encoders,
                    task=task,
                    model_name=model_name
                )
                save_ensemble_results(
                    metrics,
                    preds_traffic,
                    preds_device,
                    labels_traffic,
                    labels_device,
                    dataloader.label_encoders,
                    model_name,
                    args.multi_output,
                    task
                )
            elif model_type == 'ml':
                attack_preds, device_preds, attack_probs, device_probs = server.aggregate(X_test)
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_ml(
                    server.get_model(),
                    X_test,
                    y_test_traffic,
                    y_test_device,
                    None,  # Final evaluation
                    dataloader.label_encoders,
                    multi_output=model_config['params']['multi_output'],
                    task=task,
                    model_name=model_name
                )
                save_ensemble_results(
                    metrics,
                    preds_traffic,
                    preds_device,
                    labels_traffic,
                    labels_device,
                    dataloader.label_encoders,
                    model_name,
                    model_config['params']['multi_output'],
                    task
                )
        except Exception as e:
            print(f"Error during final aggregation/evaluation for {model_name}: {e}")
            metrics = {
                'model': model_name,
                'round': 'final',
                'error': str(e)
            }
            preds_traffic, preds_device, labels_traffic, labels_device, probs_traffic, probs_device = [], [], [], [], [], []

        # Update metrics with final model training time and average client metrics
        client_avg_metrics = {
            'train_memory_usage_MB': np.mean([m['memory_usage_MB'] for client in client_metrics.values() for m in client]) if client_metrics else 0.0,
            'train_cpu_usage_percent': np.mean([m['cpu_usage_percent'] for client in client_metrics.values() for m in client]) if client_metrics else 0.0,
            'train_energy_usage_cpu_sec': np.mean([m['energy_usage_cpu_sec'] for client in client_metrics.values() for m in client]) if client_metrics else 0.0
        }
        final_metrics = metrics.copy()
        final_metrics.update({
            'model': model_name,
            'round': 'final',
            'train_time': model_train_time,
            'test_time': test_time,
            'test_per_sample': test_time / len(X_test) if X_test.size > 0 else 0.0,
            'train_memory_usage_MB': client_avg_metrics['train_memory_usage_MB'],
            'train_cpu_usage_percent': client_avg_metrics['train_cpu_usage_percent'],
            'train_energy_usage_cpu_sec': client_avg_metrics['train_energy_usage_cpu_sec']
        })
        metrics_log.append(final_metrics)

        # Print privacy budgets for DL models
        if model_type == 'dl':
            print(f"\nPrivacy Budgets for {model_name}:")
            for entry in privacy_log:
                if entry['model'] == model_name:
                    print(
                        f"Client: {entry['client']}, "
                        f"Round: {entry['communication_round']}, "
                        f"Epoch: {entry['epoch']}, "
                        f"Epsilon: {entry['epsilon']:.2f}"
                    )

        # Save final model
        os.makedirs('models', exist_ok=True)
        os.makedirs('confusion_matrices', exist_ok=True)
        if model_type == 'dl':
            save_pytorch_model(
                server.get_model(),
                dataloader.scaler,
                dataloader.label_encoders,
                feature_columns,
                f'models/{model_name}_final.pth',
                task=task
            )
            if (args.multi_output or task == 'traffic') and len(preds_traffic) > 0:
                plot_confusion_matrix(labels_traffic, preds_traffic, f'confusion_matrices/{model_name}_traffic_final.png', dataloader.label_encoders['traffic'], 'Traffic Type')
            if (args.multi_output or task == 'device') and len(preds_device) > 0:
                plot_confusion_matrix(labels_device, preds_device, f'confusion_matrices/{model_name}_device_final.png', dataloader.label_encoders['device'], 'Device')
        elif model_type == 'tf':
            save_tf_model(server.get_model(), f'models/{model_name}_final.h5')
            if (args.multi_output or task == 'traffic') and len(preds_traffic) > 0:
                plot_confusion_matrix(labels_traffic, preds_traffic, f'confusion_matrices/{model_name}_traffic_final.png', dataloader.label_encoders['traffic'], 'Traffic Type')
            if (args.multi_output or task == 'device') and len(preds_device) > 0:
                plot_confusion_matrix(labels_device, preds_device, f'confusion_matrices/{model_name}_device_final.png', dataloader.label_encoders['device'], 'Device')
        elif model_type == 'ml':
            for client_name, client in clients.items():
                if model_config['params']['multi_output']:
                    save_ml_model(
                        client.get_model(),
                        dataloader.scaler,
                        dataloader.label_encoders,
                        feature_columns,
                        f'models/{model_name}_{client_name}_final.joblib',
                        multi_output=True
                    )
                else:
                    if task == 'traffic' or model_name.endswith('_traffic'):
                        save_ml_model(
                            client.get_model(task='traffic'),
                            dataloader.scaler,
                            dataloader.label_encoders,
                            feature_columns,
                            f'models/{model_name}_{client_name}_traffic_final.joblib',
                            multi_output=False
                        )
                    if task == 'device' or model_name.endswith('_device'):
                        save_ml_model(
                            client.get_model(task='device'),
                            dataloader.scaler,
                            dataloader.label_encoders,
                            feature_columns,
                            f'models/{model_name}_{client_name}_device_final.joblib',
                            multi_output=False
                        )
            if (args.multi_output or task == 'traffic') and len(preds_traffic) > 0:
                plot_confusion_matrix(labels_traffic, preds_traffic, f'confusion_matrices/{model_name}_traffic_final.png', dataloader.label_encoders['traffic'], 'Traffic Type')
            if (args.multi_output or task == 'device') and len(preds_device) > 0 and preds_device is not None:
                plot_confusion_matrix(labels_device, preds_device, f'confusion_matrices/{model_name}_device_final.png', dataloader.label_encoders['device'], 'Device')

    # Save metrics to CSV
    os.makedirs('metrics', exist_ok=True)
    metrics_df = pd.DataFrame(metrics_log)
    metrics_df.to_csv('metrics_final.csv', index=False)
    print("\nFinal metrics saved to 'metrics_final.csv'")

    # Save privacy budget to CSV
    if privacy_log:
        save_privacy_budget(privacy_log)

Loaded NIMLABIoT_processed.csv with 1415685 rows and 76 columns.

Null values per column:
Pck_size           0
payload_bytes      0
TCP_mss_values     0
HTTP_status        0
DHCP_options       0
                  ..
Payload_Len_Q1     0
Payload_Len_Q3     0
Payload_Len_IQR    0
Traffic Type       0
Label              0
Length: 76, dtype: int64
After dropping NaN rows, dataset has 1415685 rows.

Number of duplicate rows: 0

Column data types:
Pck_size            int64
payload_bytes       int64
TCP_mss_values      int64
HTTP_status         int64
DHCP_options        int64
                    ...  
Payload_Len_Q1      int64
Payload_Len_Q3      int64
Payload_Len_IQR     int64
Traffic Type       object
Label              object
Length: 76, dtype: object

Device type class distribution before dropping rare classes:
Label
Amcrest         240000
Nestcam         195432
Nestmini        193132
SamsungTV       186911
Smartplug       184014
Powerstrip      180904
RingDoorbell    160820
Coffeemaker  

C:\Users\danie\AppData\Local\Temp\ipykernel_8768\2445364024.py:84: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = df.groupby('Label', group_keys=False).apply(



Sampled Label value counts (50% of each):
Label
Amcrest         96000
Nestcam         78173
Nestmini        77253
SamsungTV       74764
Smartplug       73606
Powerstrip      72362
RingDoorbell    64328
Coffeemaker     29789
Name: count, dtype: int64

Original rows: 1415685, Sampled rows: 566275

Non-numeric columns: []

SMOTE disabled. Using original dataset.
Dataset has 566275 rows.
Removing 11 zero-variance features: ['STUN_method', 'MQTT_type', 'TCP_URG', 'TCP_CWR', 'DNS_opcode', 'DNS_z', 'DNS_nscount', 'Direction_ratio', 'TLS_Cipher_Suites_Len', 'TLS_Server', 'NTP_Interval']

Number of features used: 63
Features used: ['Pck_size', 'payload_bytes', 'TCP_mss_values', 'HTTP_status', 'DHCP_options', 'IP_version', 'IP_proto', 'IP_chksum', 'IP_add_count', 'TCP_sport', 'TCP_seq', 'TCP_dataofs', 'TCP_FIN', 'TCP_SYN', 'TCP_RST', 'TCP_PSH', 'TCP_ACK', 'TCP_chksum', 'TCP_window_scaling', 'UDP_sport', 'UDP_chksum', 'ICMP_type', 'ICMP_chksum', 'ICMP_seq', 'BOOTP_hlen', 'BOOTP_options', 'DNS_id

C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\opacus\privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\opacus\privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\opacus\privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production


cnn Communication Round 1/10

Training client_0


...

Client client_2 trained dt model(s). Memory: 865.16 MB, CPU: 42.10%, Energy: 120.14 CPU-sec

Training client_3


Training client_3 ML Model: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.27s/it]


Client client_3 trained dt model(s). Memory: 865.16 MB, CPU: 28.10%, Energy: 71.77 CPU-sec

Training client_4


Training client_4 ML Model: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.46s/it]


Client client_4 trained dt model(s). Memory: 865.16 MB, CPU: 26.50%, Energy: 77.48 CPU-sec
traffic unique classes: [0 1 2 3]
traffic y_probs shape: (113255, 4)
traffic expected classes: 4
traffic AUC (multiclass): 0.9999106730337637
device unique classes: [0 1 2 3 4 5 6 7]
device y_probs shape: (113255, 8)
device expected classes: 8
device AUC (multiclass): 0.9998817599525527

dt Metrics after Round 10:
+----------------------------+----------+
| Metric                     | Value    |
+============================+==========+
| traffic_precision          | 0.9992   |
+----------------------------+----------+
| traffic_recall             | 0.9992   |
+----------------------------+----------+
| traffic_f1                 | 0.9992   |
+----------------------------+----------+
| traffic_kappa              | 0.9989   |
+----------------------------+----------+
| traffic_mcc                | 0.9989   |
+----------------------------+----------+
| traffic_balanced_accuracy  | 0.9991   |
+----

# Training using DL only for 10 epochs

In [3]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, cohen_kappa_score, matthews_corrcoef
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, classification_report, confusion_matrix, hamming_loss
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from dataclasses import dataclass
from typing import List, Tuple, Dict
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from torch.nn import functional as F
from opacus import PrivacyEngine
import scienceplots
import pathlib
from tabulate import tabulate
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.models import Model
import joblib
import psutil


# NEW: Configure TensorFlow GPU settings and check environment
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"TensorFlow detected {len(physical_devices)} GPU(s): {physical_devices}")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}. Falling back to CPU.")
        os.environ['CUDA_VISIBLE_DEVICES'] = ''  # Disable GPU
else:
    print("No GPU detected. Running TensorFlow on CPU.")
print(f"TensorFlow version: {tf.__version__}")

plt.style.use(["science", "no-latex"])

@dataclass
class Parameters:
    lr: float = 0.01
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sample_rate: float = 0.2
    noise_multiplier: float = 1.5
    max_grad_norm: float = 0.5
    delta: float = 1e-4
    epochs: int = 100
    num_clients: int = 5
    batch_size: int = 128
    communication_rounds: int = 10
    conv_filters: List[int] = None
    kernel_size: int = 3
    dropout_rate: float = 0.3
    use_smote: bool = False
    use_dp: bool = True
    model_type: str = 'dl' # 'ml', 'dl', or 'both'
    multi_output: bool = True  # New flag for multi-output vs single-output

args = Parameters(conv_filters=[32, 64])
os.makedirs('classification_reports', exist_ok=True)
os.makedirs('confusion_matrices', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Sampling functions (unchanged)
def sample_50_percent_by_label(df):
    if 'Label' not in df.columns:
        print("Error: 'Label' column not found in DataFrame for sampling")
        return df
    print("Initial Label value counts before sampling:")
    print(df['Label'].value_counts())
    sampled_df = df.groupby('Label', group_keys=False).apply(
        lambda x: x.sample(frac=0.4, random_state=42)
    ).reset_index(drop=True)
    print("\nSampled Label value counts (50% of each):")
    print(sampled_df['Label'].value_counts())
    print(f"\nOriginal rows: {len(df)}, Sampled rows: {len(sampled_df)}")
    return sampled_df

def sample_50_percent_by_traffic_type(df):
    if 'Traffic Type' not in df.columns:
        print("Error: 'Traffic Type' column not found in DataFrame for sampling")
        return df
    print("Initial Traffic Type value counts before sampling:")
    print(df['Traffic Type'].value_counts())
    sampled_df = df.groupby('Traffic Type', group_keys=False).apply(
        lambda x: x.sample(frac=0.3, random_state=42)
    ).reset_index(drop=True)
    print("\nSampled Label value counts (50% of each):")
    print(sampled_df['Traffic Type'].value_counts())
    print(f"\nOriginal rows: {len(df)}, Sampled rows: {len(sampled_df)}")
    return sampled_df

def sample_by_label_with_thresholds(df, threshold_high=8000, threshold_low=4000):
    if 'Label' not in df.columns:
        print("Error: 'Label' column not found in DataFrame for sampling")
        return df
    print("Initial Label value counts before sampling:")
    print(df['Label'].value_counts())
    def custom_sample(group):
        group_size = len(group)
        if group_size > threshold_high:
            sampled = group.sample(frac=0.4, random_state=42)
        elif group_size > threshold_low:
            sampled = group.sample(frac=0.6, random_state=42)
        else:
            sampled = group
        return sampled
    sampled_df = df.groupby('Label', group_keys=False).apply(custom_sample).reset_index(drop=True)
    return sampled_df

def sample_by_Traffic_Type_with_thresholds(df, threshold_high=8000, threshold_low=4000):
    if 'Traffic Type' not in df.columns:
        print("Error: 'Traffic Type' column not found in DataFrame for sampling")
        return df
    print("Initial Traffic Type value counts before sampling:")
    print(df['Traffic Type'].value_counts())
    def custom_sample(group):
        group_size = len(group)
        if group_size > threshold_high:
            sampled = group.sample(frac=0.4, random_state=42)
        elif group_size > threshold_low:
            sampled = group.sample(frac=0.6, random_state=42)
        else:
            sampled = group
        return sampled
    sampled_df = df.groupby('Traffic Type', group_keys=False).apply(custom_sample).reset_index(drop=True)
    return sampled_df

def sample_40_percent_by_label_and_traffic(df):
    if 'Label' not in df.columns or 'Traffic Type' not in df.columns:
        missing_cols = [col for col in ['Label', 'Traffic Type'] if col not in df.columns]
        print(f"Error: {missing_cols} column(s) not found in DataFrame for sampling")
        return df
    print("Initial Label and Traffic value counts before sampling:")
    print(df.groupby(['Label', 'Traffic Type']).size().to_string())
    sampled_df = df.groupby(['Label', 'Traffic Type'], group_keys=False).apply(
        lambda x: x.sample(frac=0.4, random_state=42)
    ).reset_index(drop=True)
    print("\nSampled Label and Traffic value counts (40% of each):")
    print(sampled_df.groupby(['Label', 'Traffic Type']).size().to_string())
    print(f"\nOriginal rows: {len(df)}, Sampled rows: {len(sampled_df)}")
    return sampled_df

def sample_by_label_and_traffic_with_limits(df, min_samples=3000, max_samples=3500):
    if 'Label' not in df.columns or 'Traffic Type' not in df.columns:
        missing_cols = [col for col in ['Label', 'Traffic Type'] if col in df.columns]
        print(f"Error: {missing_cols} column(s) not found in DataFrame")
        return df
    if not isinstance(min_samples, int) or not isinstance(max_samples, int):
        print("Error: min_samples and max_samples must be integers")
        return df
    if min_samples < 0 or max_samples < 0:
        print("Error: min_samples and max_samples must be non-negative")
        return df
    if min_samples > max_samples:
        print("Error: min_samples cannot be greater than max_samples")
        return df
    print("Initial Label and Traffic Type counts before sampling:")
    initial_counts = df.groupby(['Label', 'Traffic Type']).size()
    print(initial_counts.to_string())
    def sample_group(group):
        n = len(group)
        if n < min_samples:
            print(f"Warning: Group {group.name} has {n} rows, less than min_samples ({min_samples}). Taking all rows.")
            return group
        elif n > max_samples:
            return group.sample(n=max_samples, random_state=42)
        else:
            return group.sample(n=min_samples, random_state=42)
    sampled_df = df.groupby(['Label', 'Traffic Type'], group_keys=False).apply(sample_group).reset_index(drop=True)
    print("\nSampled Label and Traffic value counts:")
    sampled_counts = sampled_df.groupby(['Label', 'Traffic Type']).size()
    print(sampled_counts.to_string())
    print(f"\nOriginal rows: {len(df)}, Sampled rows: {len(sampled_df)}")
    return sampled_df

cols_to_use = ['ARP', 'BOOTP_hlen', 'BOOTP_options', 'DHCP_options', 'DNS_ancount', 'DNS_arcount', 'DNS_id',
               'DNS_nscount', 'DNS_opcode', 'DNS_qr', 'DNS_ra', 'DNS_rd', 'DNS_z', 'Direction_ratio', 'Entropy_Min',
               'Entropy_Var', 'HTTP_method', 'HTTP_status', 'IAT_mean', 'ICMP_Chksum_Status', 'ICMP_chksum', 'ICMP_seq',
               'ICMP_type', 'IP_Dst_Entropy', 'IP_add_count', 'IP_chksum', 'IP_proto', 'IP_version', 'MQTT_type',
               'Min_Elapsed_Time', 'NTP_Interval', 'Payload_Len_IQR', 'Payload_Len_Q1', 'Payload_Len_Q3', 'Pck_Size_Avg',
               'Pck_Size_Kurt', 'Pck_Size_Max', 'Pck_Size_Med', 'Pck_Size_Min', 'Pck_Size_Q1', 'Pck_Size_Q3',
               'Pck_Size_Sum', 'Pck_Size_Var', 'Pck_size', 'Portcl_dst', 'Portcl_src', 'Protocol_ICMP_Ratio',
               'Protocol_UDP', 'Protocol_UDP_Ratio', 'STUN_method', 'TCP_ACK', 'TCP_CWR', 'TCP_FIN', 'TCP_PSH',
               'TCP_RST', 'TCP_SYN', 'TCP_URG', 'TCP_chksum', 'TCP_dataofs', 'TCP_mss_values', 'TCP_seq', 'TCP_sport',
               'TCP_window_scaling', 'TLS_Cipher_Suites_Len', 'TLS_Server', 'TTL_default', 'Time_Since_Prev_Frame',
               'UDP_chksum', 'UDP_sport', 'dport23', 'dport_bare', 'payload_bytes', 'sport23', 'sport_bare','Label', 'Traffic Type']

def load_and_preprocess_data(csv_path):
    csv_path = pathlib.Path(csv_path)
    df = pd.read_csv(csv_path, usecols=cols_to_use, low_memory=False)
    print(f"Loaded {csv_path.name} with {len(df)} rows and {len(df.columns)} columns.")
    if 'MAC' in df.columns:
        df = df.drop(columns=['MAC'])
    print("\nNull values per column:")
    print(df.isna().sum())
    df = df.fillna(0)
    print(f"After dropping NaN rows, dataset has {len(df)} rows.")
    duplicates = df.duplicated().sum()
    print(f"\nNumber of duplicate rows: {duplicates}")
    if duplicates > 0:
        df = df.drop_duplicates()
        print(f"Dropped {duplicates} duplicate rows.")
    print("\nColumn data types:")
    print(df.dtypes)
    if 'Traffic Type' not in df.columns:
        print("Warning: 'Traffic Type' column not found. Assuming binary attack category based on 'Label'.")
        df['Traffic Type'] = df['Label'].apply(lambda x: 'Attack' if 'attack' in str(x).lower() else 'Benign')
    print("\nDevice type class distribution before dropping rare classes:")
    device_counts = pd.Series(df['Label']).value_counts()
    print(device_counts)
    print("\nAttack category class distribution before dropping rare classes:")
    attack_counts = pd.Series(df['Traffic Type']).value_counts()
    print(attack_counts)
    print("\nCombined (Label, Traffic Type) distribution before dropping rare classes:")
    combined_counts = df.groupby(['Label', 'Traffic Type']).size()
    print(combined_counts)
    min_samples = 3
    rare_device_classes = device_counts[device_counts < min_samples].index
    rare_attack_classes = attack_counts[attack_counts < min_samples].index
    rare_combined = combined_counts[combined_counts < min_samples].index
    print("\nDropping device type classes with <{} samples: {}".format(min_samples, rare_device_classes.tolist()))
    print("Dropping attack category classes with <{} samples: {}".format(min_samples, rare_attack_classes.tolist()))
    print("Dropping combined (Label, Traffic Type) classes with <{} samples: {}".format(min_samples, rare_combined.tolist()))
    df = df[~df['Label'].isin(rare_device_classes)]
    df = df[~df['Traffic Type'].isin(rare_attack_classes)]
    df = df[~df[['Label', 'Traffic Type']].apply(tuple, axis=1).isin(rare_combined)]
    print(f"\nAfter dropping rare classes, dataset has {len(df)} rows.")
    print("\nDevice type class distribution after dropping rare classes:")
    device_counts = pd.Series(df['Label']).value_counts()
    print(device_counts)
    print("\nAttack category class distribution after dropping rare classes:")
    attack_counts = pd.Series(df['Traffic Type']).value_counts()
    print(attack_counts)
    columns_to_drop = ['BOOTP_flags', 'Ether_Src_OUI', 'Ether_Dst_OUI', 'Highest_Layer']
    columns_to_drop = [col for col in columns_to_drop if col in df.columns]
    df = df.drop(columns=columns_to_drop)
    df = sample_50_percent_by_label(df)
    X = df.drop(columns=['Label', 'Traffic Type'])
    y_device = df['Label']
    y_attack = df['Traffic Type']
    non_numeric_cols = X.select_dtypes(include=['object']).columns
    print(f"\nNon-numeric columns: {list(non_numeric_cols)}")
    label_encoders = {}
    for col in non_numeric_cols:
        print(f"Encoding: {col}")
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le
    if not all(X.dtypes.apply(lambda x: np.issubdtype(x, np.number))):
        print("Error: Some columns are non-numeric after encoding:")
        print(X.dtypes)
        raise ValueError("Non-numeric columns detected.")
    if np.any(np.isinf(X.to_numpy())):
        print("Warning: Inf values found in features. Replacing with max finite value.")
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.fillna(X.max().max())
    if args.use_smote:
        print("\nApplying SMOTE to balance classes...")
        if args.multi_output:
            y_combined = np.column_stack([y_device, y_attack])
            smote = SMOTE(random_state=42, k_neighbors=5)
            try:
                X_resampled, y_combined_resampled = smote.fit_resample(X, y_combined)
                y_device_resampled = y_combined_resampled[:, 0]
                y_attack_resampled = y_combined_resampled[:, 1]
                print(f"After SMOTE, dataset has {len(X_resampled)} rows.")
                print("\nDevice type distribution after SMOTE:")
                print(pd.Series(y_device_resampled).value_counts())
                print("\nAttack category distribution after SMOTE:")
                print(pd.Series(y_attack_resampled).value_counts())
            except ValueError as e:
                print(f"SMOTE failed: {e}. Disabling SMOTE.")
                X_resampled = X
                y_device_resampled = y_device
                y_attack_resampled = y_attack
        else:
            smote = SMOTE(random_state=42, k_neighbors=5)
            try:
                X_attack, y_attack_resampled = smote.fit_resample(X, y_attack)
                X_device, y_device_resampled = smote.fit_resample(X, y_device)
                print(f"After SMOTE, attack dataset: {len(X_attack)} rows, device dataset: {len(X_device)} rows.")
                X_resampled = X_attack  # Use attack data as primary; device handled separately
                y_attack_resampled = y_attack_resampled
                y_device_resampled = y_device_resampled
            except ValueError as e:
                print(f"SMOTE failed: {e}. Disabling SMOTE.")
                X_resampled = X
                y_attack_resampled = y_attack
                y_device_resampled = y_device
    else:
        print("\nSMOTE disabled. Using original dataset.")
        X_resampled = X
        y_device_resampled = y_device
        y_attack_resampled = y_attack
        print(f"Dataset has {len(X_resampled)} rows.")
    scaler = StandardScaler()
    variances = X_resampled.var()
    zero_var_cols = variances[variances == 0].index
    if len(zero_var_cols) > 0:
        print(f"Removing {len(zero_var_cols)} zero-variance features: {zero_var_cols.tolist()}")
        X_resampled = X_resampled.drop(columns=zero_var_cols)
    X_scaled = scaler.fit_transform(X_resampled)
    if np.any(np.isnan(X_scaled)) or np.any(np.isinf(X_scaled)):
        print("Error: NaN or inf in X_scaled after scaling.")
        X_scaled = np.nan_to_num(X_scaled, nan=np.nanmean(X_scaled), posinf=X_scaled.max(), neginf=X_scaled.min())
    le_device = LabelEncoder()
    le_attack = LabelEncoder()
    y_device_encoded = le_device.fit_transform(y_device_resampled)
    y_attack_encoded = le_attack.fit_transform(y_attack_resampled)
    return X_scaled, y_device_encoded, y_attack_encoded, le_device, le_attack, X_resampled.columns.tolist()

def compute_metrics(y_true, y_pred, label_type, y_probs=None, label_encoder=None):
    # Ensure y_true and y_pred only contain valid classes from label_encoder
    if label_encoder is not None:
        valid_classes = np.arange(len(label_encoder.classes_))
        valid_mask = np.isin(y_true, valid_classes) & np.isin(y_pred, valid_classes)
        y_true = y_true[valid_mask]
        y_pred = y_pred[valid_mask]
        if y_probs is not None:
            y_probs = y_probs[valid_mask]

    cm = confusion_matrix(y_true, y_pred, labels=valid_classes if label_encoder else None)
    tn = np.sum(cm) - (np.sum(cm, axis=0) + np.sum(cm, axis=1) - np.diag(cm))
    fp = np.sum(cm, axis=0) - np.diag(cm)
    fn = np.sum(cm, axis=1) - np.diag(cm)
    tp = np.diag(cm)
    tnr = np.divide(tn, (tn + fp), out=np.zeros_like(tn, dtype=float), where=(tn + fp) != 0)
    specificity = tnr
    tpr = np.divide(tp, (tp + fn), out=np.zeros_like(tp, dtype=float), where=(tp + fn) != 0)
    fpr = np.divide(fp, (fp + tn), out=np.zeros_like(fp, dtype=float), where=(fp + tn) != 0)
    fnr = np.divide(fn, (tp + fn), out=np.zeros_like(fn, dtype=float), where=(tp + fn) != 0)
    per_class_accuracy = np.divide(tp, np.sum(cm, axis=1), out=np.zeros_like(tp, dtype=float), where=np.sum(cm, axis=1) != 0)
    hamming = hamming_loss(y_true, y_pred)
    metrics = {
        f'{label_type}_precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        f'{label_type}_recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        f'{label_type}_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        f'{label_type}_kappa': cohen_kappa_score(y_true, y_pred),
        f'{label_type}_mcc': matthews_corrcoef(y_true, y_pred),
        f'{label_type}_balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        f'{label_type}_hamming_loss': hamming,
        f'{label_type}_tnr': np.mean(tnr),
        f'{label_type}_specificity': np.mean(specificity),
        f'{label_type}_tpr': np.mean(tpr),
        f'{label_type}_fpr': np.mean(fpr),
        f'{label_type}_fnr': np.mean(fnr)
    }
    if y_probs is not None:
        unique_classes = np.unique(y_true)
        expected_n_classes = len(label_encoder.classes_) if label_encoder else len(unique_classes)
        print(f"{label_type} unique classes: {unique_classes}")
        print(f"{label_type} y_probs shape: {y_probs.shape}")
        print(f"{label_type} expected classes: {expected_n_classes}")
        if y_probs.shape[1] != expected_n_classes:
            print(f"Warning: y_probs shape mismatch for {label_type}: got {y_probs.shape[1]}, expected {expected_n_classes}")
            metrics[f'{label_type}_auc'] = np.nan
        elif len(unique_classes) >= 2:
            try:
                if len(unique_classes) == 2:
                    pos_class_idx = np.max(unique_classes)
                    y_true_binary = (y_true == pos_class_idx).astype(int)
                    metrics[f'{label_type}_auc'] = roc_auc_score(y_true_binary, y_probs[:, pos_class_idx])
                    print(f"{label_type} AUC (binary): {metrics[f'{label_type}_auc']}")
                else:
                    metrics[f'{label_type}_auc'] = roc_auc_score(y_true, y_probs, multi_class='ovr')
                    print(f"{label_type} AUC (multiclass): {metrics[f'{label_type}_auc']}")
            except Exception as e:
                print(f"Warning: AUC calculation failed for {label_type}: {e}")
                metrics[f'{label_type}_auc'] = np.nan
        else:
            print(f"Warning: AUC calculation skipped for {label_type}: Only one class present")
            metrics[f'{label_type}_auc'] = np.nan
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    if label_encoder is not None:
        class_names = label_encoder.classes_
        report_dict = {}
        for i, label in enumerate(class_names):
            if str(i) in report:
                report_dict[label] = report[str(i)]
                report_dict[label]['accuracy'] = per_class_accuracy[i] if i < len(per_class_accuracy) else 0.0
        for key in ['accuracy', 'macro avg', 'weighted avg']:
            if key in report:
                report_dict[key] = report[key]
        report = report_dict
    report_df = pd.DataFrame(report).transpose()
    return metrics, report_df, cm

class DatasetLoader:
    def __init__(self, X_scaled, y_device_encoded, y_attack_encoded, le_device, le_attack, num_clients, sample_rate):
        self._num_clients = num_clients
        self._sample_rate = sample_rate
        self._label_encoder_traffic = le_attack
        self._label_encoder_device = le_device
        self._scaler = StandardScaler()
        if not (len(X_scaled) == len(y_device_encoded) == len(y_attack_encoded)):
            raise ValueError(
                f"Inconsistent input sizes: X_scaled={len(X_scaled)}, "
                f"y_device_encoded={len(y_device_encoded)}, "
                f"y_attack_encoded={len(y_attack_encoded)}"
            )
        self._X_train, self._y_train_traffic, self._y_train_device, self._X_test, self._y_test_traffic, self._y_test_device = self._load_train_test_data(
            X_scaled, y_attack_encoded, y_device_encoded
        )
        self._clients_data = self._assign_data_to_clients(self._X_train, self._y_train_traffic, self._y_train_device)

    @property
    def clients_data(self):
        return self._clients_data

    @property
    def scaler(self):
        return self._scaler

    @property
    def label_encoders(self):
        return {'traffic': self._label_encoder_traffic, 'device': self._label_encoder_device}

    def get_train_data(self):
        train_dataset = TensorDataset(
            torch.tensor(self._X_train, dtype=torch.float32).unsqueeze(1),
            torch.stack([torch.tensor(self._y_train_traffic, dtype=torch.long), torch.tensor(self._y_train_device, dtype=torch.long)], dim=1)
        )
        return DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())

    def get_test_data(self):
        test_dataset = TensorDataset(
            torch.tensor(self._X_test, dtype=torch.float32).unsqueeze(1),
            torch.stack([torch.tensor(self._y_test_traffic, dtype=torch.long), torch.tensor(self._y_test_device, dtype=torch.long)], dim=1)
        )
        return DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    def get_ml_data(self):
        return self._X_train, self._y_train_traffic, self._y_train_device, self._X_test, self._y_test_traffic, self._y_test_device

    def _load_train_test_data(self, X_scaled, y_attack_encoded, y_device_encoded):
        stratified_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        try:
            for train_idx, test_idx in stratified_split.split(X_scaled, y_attack_encoded):
                X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
                y_train_traffic, y_test_traffic = y_attack_encoded[train_idx], y_attack_encoded[test_idx]
                y_train_device, y_test_device = y_device_encoded[train_idx], y_device_encoded[test_idx]
        except ValueError as e:
            print(f"Stratified split failed: {e}. Falling back to random split.")
            indices = np.arange(len(X_scaled))
            np.random.shuffle(indices)
            split = int(0.8 * len(X_scaled))
            train_idx, test_idx = indices[:split], indices[split:]
            X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
            y_train_traffic, y_test_traffic = y_attack_encoded[train_idx], y_attack_encoded[test_idx]
            y_train_device, y_test_device = y_device_encoded[train_idx], y_device_encoded[test_idx]
        self._scaler.fit(X_train)
        X_train = self._scaler.transform(X_train)
        X_test = self._scaler.transform(X_test)
        return X_train, y_train_traffic, y_train_device, X_test, y_test_traffic, y_test_device

    def _assign_data_to_clients(self, X_train, y_train_traffic, y_train_device):
        clients_all_data = {}
        num_samples = X_train.shape[0]
        indices = np.arange(num_samples)
        np.random.seed(42)
        np.random.shuffle(indices)
        split_indices = np.array_split(indices, self._num_clients)
        for i, idx in enumerate(split_indices):
            batched_x = X_train[idx]
            batched_y_traffic = y_train_traffic[idx]
            batched_y_device = y_train_device[idx]
            x_tensor = torch.tensor(batched_x, dtype=torch.float32).unsqueeze(1)
            y_traffic_tensor = torch.tensor(batched_y_traffic, dtype=torch.long)
            y_device_tensor = torch.tensor(batched_y_device, dtype=torch.long)
            if torch.any(torch.isnan(x_tensor)) or torch.any(torch.isinf(x_tensor)):
                print(f"Warning: NaN or inf in client_{i} input data.")
                x_tensor = torch.nan_to_num(x_tensor, nan=0.0, posinf=1.0, neginf=-1.0)
            client_dataset = TensorDataset(
                x_tensor,
                torch.stack([y_traffic_tensor, y_device_tensor], dim=1)
            )
            data_loader = DataLoader(
                client_dataset,
                batch_size=args.batch_size,
                shuffle=True,
                num_workers=0,
                pin_memory=torch.cuda.is_available()
            )
            clients_all_data[f'client_{i}'] = {
                'dl_loader': data_loader,
                'X': batched_x,
                'y_traffic': batched_y_traffic,
                'y_device': batched_y_device
            }
        return clients_all_data

    def print_client_label_distribution(self):
        for client_name, data in self._clients_data.items():
            traffic_labels = data['y_traffic'].tolist()
            device_labels = data['y_device'].tolist()
            traffic_counts = {}
            device_counts = {}
            for label in traffic_labels:
                original_label = self._label_encoder_traffic.inverse_transform([label])[0]
                traffic_counts[original_label] = traffic_counts.get(original_label, 0) + 1
            for label in device_labels:
                original_label = self._label_encoder_device.inverse_transform([label])[0]
                device_counts[original_label] = device_counts.get(original_label, 0) + 1
            print(f"\n{client_name} has {len(traffic_labels)} samples:")
            print("Traffic Type distribution:")
            for label_name, count in traffic_counts.items():
                print(f"  {label_name}: {count}")
            print("Device Label distribution:")
            for label_name, count in device_counts.items():
                print(f"  {label_name}: {count}")

class MultiOutputCNN(nn.Module):
    def __init__(self, input_shape, num_classes_traffic, num_classes_device, task='both'):
        super().__init__()
        self.task = task
        self.conv1 = nn.Conv1d(1, args.conv_filters[0], kernel_size=args.kernel_size, padding=args.kernel_size//2)
        self.norm1 = nn.GroupNorm(num_groups=4, num_channels=args.conv_filters[0])
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.conv2 = nn.Conv1d(args.conv_filters[0], args.conv_filters[1], kernel_size=args.kernel_size, padding=args.kernel_size//2)
        self.norm2 = nn.GroupNorm(num_groups=8, num_channels=args.conv_filters[1])
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(args.conv_filters[1] * (input_shape // 4), 128)
        self.relu3 = nn.ReLU()
        self.dropout = nn.Dropout(p=args.dropout_rate)
        if task == 'both':
            self.fc_traffic = nn.Linear(128, num_classes_traffic)
            self.fc_device = nn.Linear(128, num_classes_device)
        elif task == 'traffic':
            self.fc_traffic = nn.Linear(128, num_classes_traffic)
        elif task == 'device':
            self.fc_device = nn.Linear(128, num_classes_device)
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight.data, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.conv1(x)
        x = self.norm1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.norm2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.dropout(x)
        if self.task == 'both':
            traffic_out = self.fc_traffic(x)
            device_out = self.fc_device(x)
            traffic_out = torch.clamp(traffic_out, min=-100, max=100)
            device_out = torch.clamp(device_out, min=-100, max=100)
            return traffic_out, device_out
        elif self.task == 'traffic':
            traffic_out = self.fc_traffic(x)
            return traffic_out
        elif self.task == 'device':
            device_out = self.fc_device(x)
            return device_out

class LSTMModel(nn.Module):
    def __init__(self, input_shape, num_classes_traffic, num_classes_device, task='both'):
        super().__init__()
        self.task = task
        self.lstm = nn.LSTM(input_size=input_shape, hidden_size=64, num_layers=2, batch_first=True)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=args.dropout_rate)
        if task == 'both':
            self.fc_traffic = nn.Linear(128, num_classes_traffic)
            self.fc_device = nn.Linear(128, num_classes_device)
        elif task == 'traffic':
            self.fc_traffic = nn.Linear(128, num_classes_traffic)
        elif task == 'device':
            self.fc_device = nn.Linear(128, num_classes_device)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight.data, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = x.squeeze(1)
        x = x.unsqueeze(1)
        out, _ = self.lstm(x)
        x = self.flatten(out[:, -1, :])
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        if self.task == 'both':
            traffic_out = self.fc_traffic(x)
            device_out = self.fc_device(x)
            return traffic_out, device_out
        elif self.task == 'traffic':
            return self.fc_traffic(x)
        elif self.task == 'device':
            return self.fc_device(x)

class BiLSTM(nn.Module):
    def __init__(self, input_shape, num_classes_traffic, num_classes_device, task='both'):
        super().__init__()
        self.task = task
        self.bilstm = nn.LSTM(input_size=input_shape, hidden_size=64, num_layers=2, batch_first=True, bidirectional=True)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 2, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=args.dropout_rate)
        if task == 'both':
            self.fc_traffic = nn.Linear(128, num_classes_traffic)
            self.fc_device = nn.Linear(128, num_classes_device)
        elif task == 'traffic':
            self.fc_traffic = nn.Linear(128, num_classes_traffic)
        elif task == 'device':
            self.fc_device = nn.Linear(128, num_classes_device)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight.data, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = x.squeeze(1)
        x = x.unsqueeze(1)
        out, _ = self.bilstm(x)
        x = self.flatten(out[:, -1, :])
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        if self.task == 'both':
            traffic_out = self.fc_traffic(x)
            device_out = self.fc_device(x)
            return traffic_out, device_out
        elif self.task == 'traffic':
            return self.fc_traffic(x)
        elif self.task == 'device':
            return self.fc_device(x)

class TF1DCNN:
    def __init__(self, input_shape, num_classes_device, num_classes_attack, task='both'):
        # NEW: Configure GPU memory growth
        physical_devices = tf.config.list_physical_devices('GPU')
        if physical_devices:
            try:
                for gpu in physical_devices:
                    tf.config.experimental.set_memory_growth(gpu, True)
            except RuntimeError as e:
                print(f"GPU memory configuration failed: {e}")

        self.task = task
        self.model = self._build_1dcnn((input_shape, 1), num_classes_device, num_classes_attack)
        self.input_shape = input_shape

    def _build_1dcnn(self, input_shape, num_classes_device, num_classes_attack):
        inputs = Input(shape=input_shape)
        x = Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
        x = MaxPooling1D(pool_size=2)(x)
        x = Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
        x = MaxPooling1D(pool_size=2)(x)
        x = Flatten()(x)
        x = Dense(128, activation='relu')(x)
        x = Dropout(0.5)(x)
        if self.task == 'both':
            device_output = Dense(num_classes_device, activation='softmax', name='device')(x)
            attack_output = Dense(num_classes_attack, activation='softmax', name='attack')(x)
            outputs = [device_output, attack_output]
        elif self.task == 'traffic':
            attack_output = Dense(num_classes_attack, activation='softmax', name='attack')(x)
            outputs = attack_output
        elif self.task == 'device':
            device_output = Dense(num_classes_device, activation='softmax', name='device')(x)
            outputs = device_output
        model = Model(inputs=inputs, outputs=outputs)
        if self.task == 'both':
            model.compile(
                optimizer='adam',
                loss={'device': 'sparse_categorical_crossentropy', 'attack': 'sparse_categorical_crossentropy'},
                metrics={'device': 'accuracy', 'attack': 'accuracy'}
            )
        else:
            model.compile(
                optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )
        return model

    def get_weights(self):
        return self.model.get_weights()

    def set_weights(self, weights):
        self.model.set_weights(weights)

    def predict(self, x):
        x = x.reshape(-1, self.input_shape, 1)
        if self.task == 'both':
            device_probs, attack_probs = self.model.predict(x, verbose=0)
            device_preds = np.argmax(device_probs, axis=1)
            attack_preds = np.argmax(attack_probs, axis=1)
            return attack_preds, device_preds, attack_probs, device_probs
        elif self.task == 'traffic':
            attack_probs = self.model.predict(x, verbose=0)
            attack_preds = np.argmax(attack_probs, axis=1)
            return attack_preds, None, attack_probs, None
        elif self.task == 'device':
            device_probs = self.model.predict(x, verbose=0)
            device_preds = np.argmax(device_probs, axis=1)
            return None, device_preds, None, device_probs

    def fit(self, X, y_traffic, y_device, epochs, batch_size):
        X = X.reshape(-1, self.input_shape, 1)
        if self.task == 'both':
            self.model.fit(
                X, {'device': y_device, 'attack': y_traffic},
                epochs=epochs, batch_size=batch_size, verbose=1
            )
        elif self.task == 'traffic':
            self.model.fit(X, y_traffic, epochs=epochs, batch_size=batch_size, verbose=1)
        elif self.task == 'device':
            self.model.fit(X, y_device, epochs=epochs, batch_size=batch_size, verbose=1)

class MLModel:
    def __init__(self, model_type, n_estimators=50, multi_output=True):
        self.multi_output = multi_output
        # Extract base model type by removing '_traffic' or '_device' suffixes
        base_model_type = model_type.replace('_traffic', '').replace('_device', '')
        if base_model_type == 'xgboost':
            base_model = XGBClassifier(n_estimators=n_estimators, random_state=42)
        elif base_model_type == 'rf':
            base_model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
        elif base_model_type == 'dt':
            base_model = DecisionTreeClassifier(random_state=42)
        else:
            raise ValueError(f"Unknown ML model type: {base_model_type}")
        if multi_output:
            self.model = MultiOutputClassifier(base_model, n_jobs=-1)
        else:
            self.model = base_model
        self.model_type = model_type

    def fit(self, X, y_traffic, y_device=None):
        if self.multi_output:
            y = np.column_stack((y_traffic, y_device))
            self.model.fit(X, y)
        else:
            self.model.fit(X, y_traffic)

    def predict(self, X):
        if self.multi_output:
            preds = self.model.predict(X)
            probs = self.model.predict_proba(X)
            attack_preds = preds[:, 0]
            device_preds = preds[:, 1]
            attack_probs = probs[0]
            device_probs = probs[1]
            return attack_preds, device_preds, attack_probs, device_probs
        else:
            preds = self.model.predict(X)
            probs = self.model.predict_proba(X)
            # Return None for device if single-output model
            return preds, None, probs, None

class MLEnsembleModel:
    def __init__(self, client_models, model_type, multi_output=True):
        self.client_models = client_models
        self.model_type = model_type
        self.multi_output = multi_output

    def predict(self, X):
        if self.multi_output:
            attack_probs_list = []
            device_probs_list = []
            for model in self.client_models:
                _, _, attack_probs, device_probs = model.predict(X)
                attack_probs_list.append(attack_probs)
                device_probs_list.append(device_probs)
            attack_probs = np.mean(attack_probs_list, axis=0)
            device_probs = np.mean(device_probs_list, axis=0)
            attack_preds = np.argmax(attack_probs, axis=1)
            device_preds = np.argmax(device_probs, axis=1)
            return attack_preds, device_preds, attack_probs, device_probs
        else:
            probs_list = []
            for model in self.client_models:
                _, _, probs, _ = model.predict(X)
                probs_list.append(probs)
            probs = np.mean(probs_list, axis=0)
            preds = np.argmax(probs, axis=1)
            return preds, None, probs, None

class DLServer:
    def __init__(self, model_class, input_size, num_classes_traffic, num_classes_device, task='both'):
        self._model = model_class(input_shape=input_size, num_classes_traffic=num_classes_traffic, num_classes_device=num_classes_device, task=task).to(args.device)
        self._client_parameters = []
        self._client_samples = []

    def add_client(self, parameters, num_samples):
        print(f"Adding client parameters with {num_samples} samples")
        self._client_parameters.append([p.cpu() for p in parameters])  # Ensure parameters are on CPU to save memory
        self._client_samples.append(num_samples)
        print(f"Current number of client parameter sets: {len(self._client_parameters)}")

    def aggregate(self):
        if not self._client_parameters:
            print("Warning: No client parameters to aggregate. Skipping aggregation.")
            return
        print(f"Aggregating {len(self._client_parameters)} client parameter sets")
        weights = [n / sum(self._client_samples) for n in self._client_samples]
        aggregated_params = [torch.zeros_like(p) for p in self._client_parameters[0]]
        for i, params in enumerate(self._client_parameters):
            for j, param in enumerate(params):
                aggregated_params[j] += weights[i] * param
        with torch.no_grad():
            for param, agg_param in zip(self._model.parameters(), aggregated_params):
                param.copy_(agg_param)
        self._client_parameters = []  # Clear after aggregation
        self._client_samples = []  # Clear sample counts
        print("Aggregation completed and client parameters cleared")

    def get_model(self):
        return self._model

class TF1Server:
    def __init__(self, input_shape, num_classes_traffic, num_classes_device, task='both'):
        self._model = TF1DCNN(input_shape, num_classes_device, num_classes_traffic, task=task)
        self._client_weights = []
        self._client_samples = []
        self.task = task

    def add_client(self, weights, num_samples):
        self._client_weights.append(weights)
        self._client_samples.append(num_samples)

    def aggregate(self):
        weights = [n / sum(self._client_samples) for n in self._client_samples]
        aggregated_weights = []
        for layer in zip(*self._client_weights):
            weighted_sum = sum(w * layer for w, layer in zip(weights, layer))
            aggregated_weights.append(weighted_sum)
        self._model.set_weights(aggregated_weights)
        self._client_weights.clear()
        self._client_samples.clear()

    def get_model(self):
        return self._model

class MLServer:
    def __init__(self, model_type, multi_output=True):
        self._models = {}
        self.model_type = model_type
        self.multi_output = multi_output
        self.ensemble_model = None

    def add_client(self, model, client_name, X, y_traffic, y_device=None):
        self._models[client_name] = (model, X, y_traffic, y_device)

    def aggregate(self, X_test):
        # Create ensemble model from client models
        client_models = [model for model, _, _, _ in self._models.values()]
        if not client_models:
            raise ValueError("No client models available for aggregation")
        self.ensemble_model = MLEnsembleModel(client_models, self.model_type, self.multi_output)
        # Return predictions from the ensemble model
        return self.ensemble_model.predict(X_test)

    def get_model(self, client_name=None):
        if client_name:
            return self._models[client_name][0]
        if self.ensemble_model is None:
            raise ValueError("Ensemble model not initialized. Call aggregate() first.")
        return self.ensemble_model

class DLClient:
    def __init__(self, model_class, data_loader, input_size, num_classes_traffic, num_classes_device, y_train_traffic, y_train_device, task='both'):
        self._model = model_class(input_shape=input_size, num_classes_traffic=num_classes_traffic, num_classes_device=num_classes_device, task=task).to(args.device)
        self._optimizer = optim.SGD(self._model.parameters(), lr=args.lr)
        self.data_loader = data_loader
        self.task = task
        try:
            unique_traffic = np.unique(y_train_traffic)
            unique_device = np.unique(y_train_device)
            class_weights_traffic = compute_class_weight('balanced', classes=unique_traffic, y=y_train_traffic)
            class_weights_device = compute_class_weight('balanced', classes=unique_device, y=y_train_device)
            if np.any(np.isnan(class_weights_traffic)) or np.any(np.isinf(class_weights_traffic)):
                class_weights_traffic = np.ones(len(unique_traffic))
            if np.any(np.isnan(class_weights_device)) or np.any(np.isinf(class_weights_device)):
                class_weights_device = np.ones(len(unique_device))
        except:
            class_weights_traffic = np.ones(num_classes_traffic)
            class_weights_device = np.ones(num_classes_device)
        self._loss_traffic = nn.CrossEntropyLoss(weight=torch.tensor(class_weights_traffic, dtype=torch.float32).to(args.device))
        self._loss_device = nn.CrossEntropyLoss(weight=torch.tensor(class_weights_device, dtype=torch.float32).to(args.device))
        if args.use_dp:
            self.privacy_engine = PrivacyEngine()
            try:
                self._model, self._optimizer, self.data_loader = self.privacy_engine.make_private(
                    module=self._model,
                    optimizer=self._optimizer,
                    data_loader=self.data_loader,
                    noise_multiplier=args.noise_multiplier,
                    max_grad_norm=args.max_grad_norm
                )
            except Exception as e:
                print(f"PrivacyEngine failed: {e}. Disabling DP.")
                args.use_dp = False

    def train(self, epochs, client_name):
        self._model.train()
        process = psutil.Process()
        epsilons = []
        memory_usages = []
        cpu_usages = []
        energy_usages = []
        for epoch in range(epochs):
            losses_traffic = []
            losses_device = []
            epoch_start_time = time.time()
            cpu_percent = []
            for batch_idx, (x, y) in enumerate(tqdm(self.data_loader, desc=f'Training {client_name} Epoch {epoch+1}')):
                x = x.to(args.device).to(torch.float32)
                if self.task == 'both':
                    y_traffic, y_device = y[:, 0].to(args.device), y[:, 1].to(args.device)
                    self._optimizer.zero_grad()
                    traffic_out, device_out = self._model(x)
                    loss_traffic = self._loss_traffic(traffic_out, y_traffic)
                    loss_device = self._loss_device(device_out, y_device)
                    loss = loss_traffic + loss_device
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self._model.parameters(), max_norm=1.0)
                    self._optimizer.step()
                    losses_traffic.append(loss_traffic.item())
                    losses_device.append(loss_device.item())
                else:
                    y_target = y[:, 0 if self.task == 'traffic' else 1].to(args.device)
                    self._optimizer.zero_grad()
                    out = self._model(x)
                    loss = self._loss_traffic(out, y_target) if self.task == 'traffic' else self._loss_device(out, y_target)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self._model.parameters(), max_norm=1.0)
                    self._optimizer.step()
                    (losses_traffic if self.task == 'traffic' else losses_device).append(loss.item())
                cpu_percent.append(psutil.cpu_percent(interval=None))
            memory_usage = process.memory_info().rss / (1024 ** 2)  # MB
            cpu_usage = np.mean(cpu_percent) if cpu_percent else 0.0
            epoch_time = time.time() - epoch_start_time
            energy_usage = cpu_usage * epoch_time  # Proxy: CPU% * time (seconds)
            memory_usages.append(memory_usage)
            cpu_usages.append(cpu_usage)
            energy_usages.append(energy_usage)
            epsilon = self.privacy_engine.get_epsilon(args.delta) if args.use_dp else 0.0
            epsilons.append(epsilon)
            print(
                f"Client: {client_name} Epoch: {epoch+1} "
                f"Traffic Loss: {np.mean(losses_traffic):.4f} "
                f"Device Loss: {np.mean(losses_device):.4f} "
                f"Memory: {memory_usage:.2f} MB "
                f"CPU: {cpu_usage:.2f}% "
                f"Energy: {energy_usage:.2f} CPU-sec "
                f"| (ε = {epsilon:.2f})"
            )
        return epsilons, {
            'memory_usage_MB': np.mean(memory_usages),
            'cpu_usage_percent': np.mean(cpu_usages),
            'energy_usage_cpu_sec': np.mean(energy_usages)
        }

    def get_parameters(self):
        return list(self._model.parameters())

    def set_parameters(self, parameters):
        with torch.no_grad():
            for param, new_param in zip(self._model.parameters(), parameters):
                param.data = new_param.data.clone()

    def get_model(self):
        return self._model

class TF1Client:
    def __init__(self, X, y_traffic, y_device, input_shape, num_classes_traffic, num_classes_device, task='both'):
        self._model = TF1DCNN(input_shape, num_classes_device, num_classes_traffic, task=task)
        self.X = X
        self.y_traffic = y_traffic
        self.y_device = y_device
        self.task = task

    def train(self, epochs, client_name):
        process = psutil.Process()
        start_time = time.time()
        cpu_percent = []
        for _ in tqdm(range(epochs), desc=f'Training {client_name} TF1 Epoch'):
            if self.task == 'both':
                self._model.fit(self.X, self.y_traffic, self.y_device, epochs=1, batch_size=args.batch_size)
            elif self.task == 'traffic':
                self._model.fit(self.X, self.y_traffic, None, epochs=1, batch_size=args.batch_size)
            elif self.task == 'device':
                self._model.fit(self.X, None, self.y_device, epochs=1, batch_size=args.batch_size)
            cpu_percent.append(psutil.cpu_percent(interval=None))
        memory_usage = process.memory_info().rss / (1024 ** 2)  # MB
        cpu_usage = np.mean(cpu_percent) if cpu_percent else 0.0
        train_time = time.time() - start_time
        energy_usage = cpu_usage * train_time  # Proxy: CPU% * time (seconds)
        print(
            f"Client {client_name} trained for {epochs} epochs. "
            f"Memory: {memory_usage:.2f} MB, "
            f"CPU: {cpu_usage:.2f}%, "
            f"Energy: {energy_usage:.2f} CPU-sec"
        )
        return {
            'memory_usage_MB': memory_usage,
            'cpu_usage_percent': cpu_usage,
            'energy_usage_cpu_sec': energy_usage
        }

    def get_parameters(self):
        return self._model.get_weights()

    def set_parameters(self, weights):
        self._model.set_weights(weights)

    def get_model(self):
        return self._model

class MLClient:
    def __init__(self, model_type, X, y_traffic, y_device, multi_output=True):
        self.multi_output = multi_output
        if multi_output:
            self._model = MLModel(model_type, multi_output=True)
        else:
            base_model_type = model_type.replace('_traffic', '').replace('_device', '')
            self._model_traffic = MLModel(f"{base_model_type}_traffic", multi_output=False)
            self._model_device = MLModel(f"{base_model_type}_device", multi_output=False)
        self.X = X
        self.y_traffic = y_traffic
        self.y_device = y_device

    def train(self, epochs=None, client_name=None):
        process = psutil.Process()
        epochs = 1 if epochs is None else epochs
        start_time = time.time()
        cpu_percent = []
        for _ in tqdm(range(epochs), desc=f"Training {client_name} ML Model"):
            if self.multi_output:
                self._model.fit(self.X, self.y_traffic, self.y_device)
            else:
                self._model_traffic.fit(self.X, self.y_traffic)
                self._model_device.fit(self.X, self.y_device)
            cpu_percent.append(psutil.cpu_percent(interval=None))
        memory_usage = process.memory_info().rss / (1024 ** 2)  # MB
        cpu_usage = np.mean(cpu_percent) if cpu_percent else 0.0
        train_time = time.time() - start_time
        energy_usage = cpu_usage * train_time  # Proxy: CPU% * time (seconds)
        print(
            f"Client {client_name} trained {self._model.model_type if self.multi_output else f'{self._model_traffic.model_type}, {self._model_device.model_type}'} model(s). "
            f"Memory: {memory_usage:.2f} MB, "
            f"CPU: {cpu_usage:.2f}%, "
            f"Energy: {energy_usage:.2f} CPU-sec"
        )
        return {
            'memory_usage_MB': memory_usage,
            'cpu_usage_percent': cpu_usage,
            'energy_usage_cpu_sec': energy_usage
        }

    def get_model(self, task='both'):
        if self.multi_output:
            return self._model
        return self._model_traffic if task == 'traffic' else self._model_device

def save_ensemble_results(metrics, preds_traffic, preds_device, labels_traffic, labels_device, label_encoders, model_name, multi_output, task):
    """
    Save and print ensemble results (metrics, classification report, confusion matrix) for ML models.
    """
    # Save metrics
    os.makedirs('metrics', exist_ok=True)
    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(f'metrics/{model_name}_ensemble_final.csv', index=False)
    print(f"\nEnsemble metrics saved for {model_name} to 'metrics/{model_name}_ensemble_final.csv'")

    # Print key metrics
    print(f"\n{model_name} Ensemble Metrics:")
    table_data = [[k, f"{v:.4f}" if isinstance(v, (float, np.floating)) else v] for k, v in metrics.items()]
    print(tabulate(table_data, headers=['Metric', 'Value'], tablefmt='grid'))

    # Save and print traffic results
    if task in ['both', 'traffic']:
        traffic_metrics, traffic_report, traffic_cm = compute_metrics(
            labels_traffic, preds_traffic, 'traffic', None, label_encoders['traffic']
        )
        os.makedirs('classification_reports', exist_ok=True)
        os.makedirs('confusion_matrices', exist_ok=True)
        traffic_report.to_csv(f"classification_reports/traffic_{model_name}_ensemble_final.csv")
        traffic_cm_df = pd.DataFrame(
            traffic_cm,
            index=label_encoders['traffic'].classes_[:traffic_cm.shape[0]],
            columns=label_encoders['traffic'].classes_[:traffic_cm.shape[1]]
        )
        traffic_cm_df.to_csv(f"confusion_matrices/traffic_{model_name}_ensemble_final.csv")
        print(f"\n{model_name} Traffic Ensemble Classification Report:")
        print(traffic_report.to_string())
        print(f"\n{model_name} Traffic Ensemble Confusion Matrix:")
        print(traffic_cm_df.to_string())

    # Save and print device results
    if multi_output or task == 'device':
        device_metrics, device_report, device_cm = compute_metrics(
            labels_device, preds_device, 'device', None, label_encoders['device']
        )
        os.makedirs('classification_reports', exist_ok=True)
        os.makedirs('confusion_matrices', exist_ok=True)
        device_report.to_csv(f"classification_reports/device_{model_name}_ensemble_final.csv")
        device_cm_df = pd.DataFrame(
            device_cm,
            index=label_encoders['device'].classes_[:device_cm.shape[0]],
            columns=label_encoders['device'].classes_[:device_cm.shape[1]]
        )
        device_cm_df.to_csv(f"confusion_matrices/device_{model_name}_ensemble_final.csv")
        print(f"\n{model_name} Device Ensemble Classification Report:")
        print(device_report.to_string())
        print(f"\n{model_name} Device Ensemble Confusion Matrix:")
        print(device_cm_df.to_string())

def evaluate_dl(model, device, test_loader, round_num=None, label_encoders=None, task='both', model_name='unknown'):
    model.eval()
    model.to(device)
    process = psutil.Process()
    start_time = time.time()
    cpu_percent = []
    loss_traffic = nn.CrossEntropyLoss()
    loss_device = nn.CrossEntropyLoss()
    test_loss_traffic = 0
    test_loss_device = 0
    correct_traffic = 0
    correct_device = 0
    all_preds_traffic = []
    all_labels_traffic = []
    all_probs_traffic = []
    all_preds_device = []
    all_labels_device = []
    all_probs_device = []
    with torch.no_grad():
        for x, y in tqdm(test_loader, desc="Evaluating"):
            x = x.to(device).to(torch.float32)
            cpu_percent.append(psutil.cpu_percent(interval=None))
            if task == 'both':
                y_traffic, y_device = y[:, 0].to(device), y[:, 1].to(device)
                traffic_out, device_out = model(x)
                test_loss_traffic += loss_traffic(traffic_out, y_traffic).item()
                test_loss_device += loss_device(device_out, y_device).item()
                probs_traffic = torch.softmax(traffic_out, dim=1)
                probs_device = torch.softmax(device_out, dim=1)
                preds_traffic = torch.argmax(probs_traffic, dim=1)
                preds_device = torch.argmax(probs_device, dim=1)
                correct_traffic += preds_traffic.eq(y_traffic).sum().item()
                correct_device += preds_device.eq(y_device).sum().item()
                all_preds_traffic.extend(preds_traffic.cpu().numpy())
                all_labels_traffic.extend(y_traffic.cpu().numpy())
                all_probs_traffic.extend(probs_traffic.cpu().numpy())
                all_preds_device.extend(preds_device.cpu().numpy())
                all_labels_device.extend(y_device.cpu().numpy())
                all_probs_device.extend(probs_device.cpu().numpy())
            else:
                y_target = y[:, 0 if task == 'traffic' else 1].to(device)
                out = model(x)
                loss = loss_traffic(out, y_target) if task == 'traffic' else loss_device(out, y_target)
                probs = torch.softmax(out, dim=1)
                preds = torch.argmax(probs, dim=1)
                correct = preds.eq(y_target).sum().item()
                if task == 'traffic':
                    test_loss_traffic += loss.item()
                    correct_traffic += correct
                    all_preds_traffic.extend(preds.cpu().numpy())
                    all_labels_traffic.extend(y_target.cpu().numpy())
                    all_probs_traffic.extend(probs.cpu().numpy())
                else:
                    test_loss_device += loss.item()
                    correct_device += correct
                    all_preds_device.extend(preds.cpu().numpy())
                    all_labels_device.extend(y_target.cpu().numpy())
                    all_probs_device.extend(probs.cpu().numpy())
    memory_usage = process.memory_info().rss / (1024 ** 2)  # MB
    cpu_usage = np.mean(cpu_percent) if cpu_percent else 0.0
    test_time = time.time() - start_time
    energy_usage = cpu_usage * test_time  # Proxy: CPU% * time (seconds)
    test_loss_traffic /= len(test_loader.dataset)
    test_loss_device /= len(test_loader.dataset)
    accuracy_traffic = correct_traffic / len(test_loader.dataset) if task in ['both', 'traffic'] else 0
    accuracy_device = correct_device / len(test_loader.dataset) if task in ['both', 'device'] else 0
    metrics = {}
    if task in ['both', 'traffic']:
        traffic_metrics, traffic_report, traffic_cm = compute_metrics(
            np.array(all_labels_traffic), np.array(all_preds_traffic), 'traffic', np.array(all_probs_traffic), label_encoders['traffic']
        )
        metrics.update(traffic_metrics)
        report_path = f"classification_reports/traffic_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/traffic_{model_name}_final.csv"
        traffic_report.to_csv(report_path)
        traffic_cm_df = pd.DataFrame(
            traffic_cm,
            index=label_encoders['traffic'].classes_[:traffic_cm.shape[0]],
            columns=label_encoders['traffic'].classes_[:traffic_cm.shape[0]]
        )
        traffic_cm_df.to_csv(f"confusion_matrices/traffic_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/traffic_{model_name}_final.csv")
    if task in ['both', 'device']:
        device_metrics, device_report, device_cm = compute_metrics(
            np.array(all_labels_device), np.array(all_preds_device), 'device', np.array(all_probs_device), label_encoders['device']
        )
        metrics.update(device_metrics)
        report_path = f"classification_reports/device_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/device_{model_name}_final.csv"
        device_cm_df = pd.DataFrame(
            device_cm,
            index=label_encoders['device'].classes_[:device_cm.shape[0]],
            columns=label_encoders['device'].classes_[:device_cm.shape[0]]
        )
        device_cm_df.to_csv(f"confusion_matrices/device_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/device_{model_name}_final.csv")
    metrics.update({
        'traffic_accuracy': accuracy_traffic,
        'device_accuracy': accuracy_device,
        'traffic_loss': test_loss_traffic,
        'device_loss': test_loss_device,
        'test_memory_usage_MB': memory_usage,
        'test_cpu_usage_percent': cpu_usage,
        'test_energy_usage_cpu_sec': energy_usage
    })
    return metrics, (all_preds_traffic, all_preds_device), (all_labels_traffic, all_labels_device), (all_probs_traffic, all_probs_device)


def evaluate_tf1(model, X_test, y_traffic_test, y_device_test, round_num=None, label_encoders=None, task='both', model_name='unknown'):
    process = psutil.Process()
    start_time = time.time()
    cpu_percent = []
    attack_preds, device_preds, attack_probs, device_probs = model.predict(X_test)
    cpu_percent.append(psutil.cpu_percent(interval=None))
    memory_usage = process.memory_info().rss / (1024 ** 2)  # MB
    cpu_usage = np.mean(cpu_percent) if cpu_percent else 0.0
    test_time = time.time() - start_time
    energy_usage = cpu_usage * test_time  # Proxy: CPU% * time (seconds)
    metrics = {}
    if task in ['both', 'traffic']:
        traffic_metrics, traffic_report, traffic_cm = compute_metrics(
            y_traffic_test, attack_preds, 'traffic', attack_probs, label_encoders['traffic']
        )
        metrics.update(traffic_metrics)
        report_path = f"classification_reports/traffic_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/traffic_{model_name}_final.csv"
        traffic_report.to_csv(report_path)
        traffic_cm_df = pd.DataFrame(
            traffic_cm,
            index=label_encoders['traffic'].classes_[:traffic_cm.shape[0]],
            columns=label_encoders['traffic'].classes_[:traffic_cm.shape[0]]
        )
        traffic_cm_df.to_csv(f"confusion_matrices/traffic_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/traffic_{model_name}_final.csv")
    if task in ['both', 'device']:
        device_metrics, device_report, device_cm = compute_metrics(
            y_device_test, device_preds, 'device', device_probs, label_encoders['device']
        )
        metrics.update(device_metrics)
        report_path = f"classification_reports/device_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/device_{model_name}_final.csv"
        device_cm_df = pd.DataFrame(
            device_cm,
            index=label_encoders['device'].classes_[:device_cm.shape[0]],
            columns=label_encoders['device'].classes_[:device_cm.shape[0]]
        )
        device_cm_df.to_csv(f"confusion_matrices/device_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/device_{model_name}_final.csv")
    metrics.update({
        'traffic_accuracy': np.mean(attack_preds == y_traffic_test) if task in ['both', 'traffic'] else 0.0,
        'device_accuracy': np.mean(device_preds == y_device_test) if task in ['both', 'device'] else 0.0,
        'traffic_loss': 0.0,
        'device_loss': 0.0,
        'test_memory_usage_MB': memory_usage,
        'test_cpu_usage_percent': cpu_usage,
        'test_energy_usage_cpu_sec': energy_usage
    })
    return metrics, (attack_preds, device_preds), (y_traffic_test, y_device_test), (attack_probs, device_probs)


def evaluate_ml(model, X_test, y_traffic_test, y_device_test, round_num=None, label_encoders=None, multi_output=True, task='both', model_name='unknown'):
    process = psutil.Process()
    start_time = time.time()
    cpu_percent = []
    attack_preds, device_preds, attack_probs, device_probs = model.predict(X_test)
    cpu_percent.append(psutil.cpu_percent(interval=None))
    memory_usage = process.memory_info().rss / (1024 ** 2)  # MB
    cpu_usage = np.mean(cpu_percent) if cpu_percent else 0.0
    test_time = time.time() - start_time
    energy_usage = cpu_usage * test_time  # Proxy: CPU% * time (seconds)
    metrics = {}
    if multi_output:
        if task in ['both', 'traffic']:
            traffic_metrics, traffic_report, traffic_cm = compute_metrics(
                y_traffic_test, attack_preds, 'traffic', attack_probs, label_encoders['traffic']
            )
            metrics.update(traffic_metrics)
            report_path = f"classification_reports/traffic_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/traffic_{model_name}_final.csv"
            traffic_report.to_csv(report_path)
            traffic_cm_df = pd.DataFrame(
                traffic_cm,
                index=label_encoders['traffic'].classes_[:traffic_cm.shape[0]],
                columns=label_encoders['traffic'].classes_[:traffic_cm.shape[0]]
            )
            traffic_cm_df.to_csv(f"confusion_matrices/traffic_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/traffic_{model_name}_final.csv")
        if task in ['both', 'device']:
            device_metrics, device_report, device_cm = compute_metrics(
                y_device_test, device_preds, 'device', device_probs, label_encoders['device']
            )
            metrics.update(device_metrics)
            report_path = f"classification_reports/device_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/device_{model_name}_final.csv"
            device_cm_df = pd.DataFrame(
                device_cm,
                index=label_encoders['device'].classes_[:device_cm.shape[0]],
                columns=label_encoders['device'].classes_[:device_cm.shape[0]]
            )
            device_cm_df.to_csv(f"confusion_matrices/device_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/device_{model_name}_final.csv")
        metrics['traffic_accuracy'] = np.mean(attack_preds == y_traffic_test) if task in ['both', 'traffic'] else 0.0
        metrics['device_accuracy'] = np.mean(device_preds == y_device_test) if task in ['both', 'device'] else 0.0
    else:
        target = 'traffic' if model.model_type.endswith('_traffic') else 'device'
        target_labels = y_traffic_test if target == 'traffic' else y_device_test
        target_probs = attack_probs
        target_encoder = label_encoders[target]
        target_metrics, target_report, target_cm = compute_metrics(
            target_labels, attack_preds, target, target_probs, target_encoder
        )
        metrics.update(target_metrics)
        report_path = f"classification_reports/{target}_{model_name}_round_{round_num}.csv" if round_num else f"classification_reports/{target}_{model_name}_final.csv"
        target_report.to_csv(report_path)
        target_cm_df = pd.DataFrame(
            target_cm,
            index=target_encoder.classes_[:target_cm.shape[0]],
            columns=target_encoder.classes_[:target_cm.shape[0]]
        )
        target_cm_df.to_csv(f"confusion_matrices/{target}_{model_name}_round_{round_num}.csv" if round_num else f"confusion_matrices/{target}_{model_name}_final.csv")
        metrics[f'{target}_accuracy'] = np.mean(attack_preds == target_labels)
        metrics['traffic_accuracy'] = metrics.get('traffic_accuracy', 0.0) if target == 'traffic' else 0.0
        metrics['device_accuracy'] = metrics.get('device_accuracy', 0.0) if target == 'device' else 0.0
    metrics.update({
        'traffic_loss': 0.0,
        'device_loss': 0.0,
        'test_memory_usage_MB': memory_usage,
        'test_cpu_usage_percent': cpu_usage,
        'test_energy_usage_cpu_sec': energy_usage
    })
    return metrics, (attack_preds, device_preds), (y_traffic_test, y_device_test), (attack_probs, device_probs)


def save_pytorch_model(model, scaler, label_encoders, feature_columns, filepath, task='both'):
    if not args.multi_output and task == 'both':
        # Save two separate models for single-output mode
        torch.save({
            'model_state_dict': model.state_dict(),
            'scaler': scaler,
            'label_encoder_traffic_classes': label_encoders['traffic'].classes_,
            'label_encoder_device_classes': [],
            'feature_columns': feature_columns,
            'task': 'traffic'
        }, filepath.replace('.pth', '_traffic.pth'))
        torch.save({
            'model_state_dict': model.state_dict(),
            'scaler': scaler,
            'label_encoder_traffic_classes': [],
            'label_encoder_device_classes': label_encoders['device'].classes_,
            'feature_columns': feature_columns,
            'task': 'device'
        }, filepath.replace('.pth', '_device.pth'))
    else:
        # Save single model for multi-output or specific task
        torch.save({
            'model_state_dict': model.state_dict(),
            'scaler': scaler,
            'label_encoder_traffic_classes': label_encoders['traffic'].classes_ if task in ['both', 'traffic'] else [],
            'label_encoder_device_classes': label_encoders['device'].classes_ if task in ['both', 'device'] else [],
            'feature_columns': feature_columns,
            'task': task
        }, filepath)

def save_tf_model(model, filepath):
    if not args.multi_output and model.task == 'both':
        # Save two separate models for single-output mode
        model.model.save(filepath.replace('.h5', '_traffic.h5'))
        model.model.save(filepath.replace('.h5', '_device.h5'))
    else:
        # Save single model for multi-output or specific task
        model.model.save(filepath)

def save_privacy_budget(privacy_log, filepath='privacy_budget.csv'):
    privacy_df = pd.DataFrame(privacy_log)
    privacy_df.to_csv(filepath, index=False)
    print(f"\nPrivacy budget saved to '{filepath}'")

def save_ml_model(model, scaler, label_encoders, feature_columns, filepath, multi_output=True):
    joblib.dump({
        'model': model,
        'scaler': scaler,
        'label_encoders': label_encoders,
        'feature_columns': feature_columns,
        'multi_output': multi_output
    }, filepath)

def plot_confusion_matrix(labels, predictions, savefile, label_encoder, label_type):
    cm = confusion_matrix(labels, predictions)
    class_names = label_encoder.classes_
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix ({label_type})")
    plt.savefig(savefile)
    plt.close()


# if __name__ == "__main__":
#     csv_file = r"F:/Datasets_Project/NIM LAB IoT Dataset 2025/slowloriss/small/processed/NIMLABIoT_processed.csv"
#     X_scaled, y_device_encoded, y_attack_encoded, le_device, le_attack, feature_columns = load_and_preprocess_data(csv_file)
#     print(f"\nNumber of features used: {len(feature_columns)}")
#     print("Features used:", feature_columns)
#     dataloader = DatasetLoader(X_scaled, y_device_encoded, y_attack_encoded, le_device, le_attack,
#                               num_clients=args.num_clients, sample_rate=args.sample_rate)
#     dataloader.print_client_label_distribution()
#     train_loader = dataloader.get_train_data()
#     test_loader = dataloader.get_test_data()
#     X_train, y_train_traffic, y_train_device, X_test, y_test_traffic, y_test_device = dataloader.get_ml_data()
#     input_shape = X_scaled.shape[1]
#     num_classes_traffic = len(np.unique(y_attack_encoded))
#     num_classes_device = len(np.unique(y_device_encoded))
#     clients_data = dataloader.clients_data
#     local_epochs = max(1, args.epochs // args.communication_rounds)

#     models = []
#     if args.model_type in ['dl', 'both']:
#         if args.multi_output:
#             models.extend([
#                 {'name': 'cnn', 'type': 'dl', 'class': MultiOutputCNN, 'server': DLServer, 'task': 'both'},
#                 {'name': 'lstm', 'type': 'dl', 'class': LSTMModel, 'server': DLServer, 'task': 'both'},
#                 {'name': 'bilstm', 'type': 'dl', 'class': BiLSTM, 'server': DLServer, 'task': 'both'},
#                 {'name': 'tf1cnn', 'type': 'tf', 'class': TF1DCNN, 'server': TF1Server, 'task': 'both'}
#             ])
#         else:
#             models.extend([
#                 {'name': 'cnn_traffic', 'type': 'dl', 'class': MultiOutputCNN, 'server': DLServer, 'task': 'traffic'},
#                 {'name': 'cnn_device', 'type': 'dl', 'class': MultiOutputCNN, 'server': DLServer, 'task': 'device'},
#                 {'name': 'lstm_traffic', 'type': 'dl', 'class': LSTMModel, 'server': DLServer, 'task': 'traffic'},
#                 {'name': 'lstm_device', 'type': 'dl', 'class': LSTMModel, 'server': DLServer, 'task': 'device'},
#                 {'name': 'bilstm_traffic', 'type': 'dl', 'class': BiLSTM, 'server': DLServer, 'task': 'traffic'},
#                 {'name': 'bilstm_device', 'type': 'dl', 'class': BiLSTM, 'server': DLServer, 'task': 'device'},
#                 {'name': 'tf1cnn_traffic', 'type': 'tf', 'class': TF1DCNN, 'server': TF1Server, 'task': 'traffic'},
#                 {'name': 'tf1cnn_device', 'type': 'tf', 'class': TF1DCNN, 'server': TF1Server, 'task': 'device'}
#             ])
#     if args.model_type in ['ml', 'both']:
#         if args.multi_output:
#             models.extend([
#                 {'name': 'xgboost', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'xgboost', 'multi_output': True}},
#                 {'name': 'rf', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'rf', 'multi_output': True}},
#                 {'name': 'dt', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'dt', 'multi_output': True}}
#             ])
#         else:
#             models.extend([
#                 {'name': 'xgboost_traffic', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'xgboost_traffic', 'multi_output': False}},
#                 {'name': 'xgboost_device', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'xgboost_device', 'multi_output': False}},
#                 {'name': 'rf_traffic', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'rf_traffic', 'multi_output': False}},
#                 {'name': 'rf_device', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'rf_device', 'multi_output': False}},
#                 {'name': 'dt_traffic', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'dt_traffic', 'multi_output': False}},
#                 {'name': 'dt_device', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'dt_device', 'multi_output': False}}
#             ])

#     metrics_log = []
#     privacy_log = []

#     for model_config in models:
#         model_name = model_config['name']
#         model_type = model_config['type']
#         task = model_config.get('task', 'both')
#         print(f"\nTraining model: {model_name}")
#         model_train_start_time = time.time()

#         if model_type == 'dl':
#             server = model_config['server'](
#                 model_class=model_config['class'],
#                 input_size=input_shape,
#                 num_classes_traffic=num_classes_traffic,
#                 num_classes_device=num_classes_device,
#                 task=task
#             )
#             clients = {
#                 client_name: DLClient(
#                     model_class=model_config['class'],
#                     data_loader=client_data['dl_loader'],
#                     input_size=input_shape,
#                     num_classes_traffic=num_classes_traffic,
#                     num_classes_device=num_classes_device,
#                     y_train_traffic=client_data['y_traffic'],
#                     y_train_device=client_data['y_device'],
#                     task=task
#                 )
#                 for client_name, client_data in clients_data.items()
#             }
#         elif model_type == 'tf':
#             server = model_config['server'](
#                 input_shape=input_shape,
#                 num_classes_traffic=num_classes_traffic,
#                 num_classes_device=num_classes_device,
#                 task=task
#             )
#             clients = {
#                 client_name: TF1Client(
#                     X=client_data['X'],
#                     y_traffic=client_data['y_traffic'],
#                     y_device=client_data['y_device'],
#                     input_shape=input_shape,
#                     num_classes_traffic=num_classes_traffic,
#                     num_classes_device=num_classes_device,
#                     task=task
#                 )
#                 for client_name, client_data in clients_data.items()
#             }
#         elif model_type == 'ml':
#             server = model_config['server'](**model_config['params'])
#             clients = {
#                 client_name: MLClient(
#                     model_type=model_config['params']['model_type'],
#                     X=client_data['X'],
#                     y_traffic=client_data['y_traffic'],
#                     y_device=client_data['y_device'],
#                     multi_output=model_config['params']['multi_output']
#                 )
#                 for client_name, client_data in clients_data.items()
#             }

#         client_metrics = {}  # Store per-client training metrics
#         for comm_round in range(args.communication_rounds):
#             print(f"\n{model_name} Communication Round {comm_round + 1}/{args.communication_rounds}")
#             round_train_start_time = time.time()
#             round_train_metrics = {
#                 'memory_usage_MB': [],
#                 'cpu_usage_percent': [],
#                 'energy_usage_cpu_sec': []
#             }

#             for client_name, client in clients.items():
#                 print(f"\nTraining {client_name}")
#                 if model_type == 'dl':
#                     client.set_parameters(server.get_model().parameters())
#                     epsilons, train_metrics = client.train(local_epochs, client_name)
#                     server.add_client(client.get_parameters(), len(client.data_loader.dataset))
#                     for epoch, epsilon in enumerate(epsilons, 1):
#                         privacy_log.append({
#                             'model': model_name,
#                             'client': client_name,
#                             'communication_round': comm_round + 1,
#                             'epoch': epoch,
#                             'epsilon': epsilon
#                         })
#                 elif model_type == 'tf':
#                     client.set_parameters(server.get_model().get_weights())
#                     train_metrics = client.train(local_epochs, client_name)
#                     server.add_client(client.get_parameters(), len(client.X))
#                 elif model_type == 'ml':
#                     train_metrics = client.train(local_epochs, client_name)
#                     if model_config['params']['multi_output']:
#                         server.add_client(client.get_model(), client_name, client.X, client.y_traffic, client.y_device)
#                     else:
#                         if task == 'traffic' or model_name.endswith('_traffic'):
#                             server.add_client(client.get_model(task='traffic'), client_name, client.X, client.y_traffic, None)
#                         if task == 'device' or model_name.endswith('_device'):
#                             server.add_client(client.get_model(task='device'), client_name, client.X, None, client.y_device)
#                 client_metrics[client_name] = client_metrics.get(client_name, []) + [train_metrics]
#                 for key in round_train_metrics:
#                     round_train_metrics[key].append(train_metrics[key])

#             # Aggregate and evaluate
#             test_start_time = time.time()
#             if model_type == 'ml' and task == 'both':
#                 attack_preds, device_preds, attack_probs, device_probs = server.aggregate(X_test)
#                 metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_ml(
#                     server.get_model(),
#                     X_test,
#                     y_test_traffic,
#                     y_test_traffic,
#                     comm_round + 1,
#                     dataloader.label_encoders,
#                     multi_output=model_config['params']['multi_output'],
#                     task=task,
#                     model_name=model_name
#                 )
#             elif model_type == 'ml':
#                 attack_preds, device_preds, attack_probs, device_probs = server.aggregate(X_test)
#                 metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_ml(
#                     server.get_model(),
#                     X_test,
#                     y_test_traffic,
#                     y_test_device,
#                     comm_round + 1,
#                     dataloader.label_encoders,
#                     multi_output=model_config['params']['multi_output'],
#                     task=task,
#                     model_name=model_name
#                 )
#             elif model_type == 'dl':
#                 server.aggregate()
#                 metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_dl(
#                     server.get_model(),
#                     args.device,
#                     test_loader,
#                     comm_round + 1,
#                     dataloader.label_encoders,
#                     task=task,
#                     model_name=model_name
#                 )
#             elif model_type == 'tf':
#                 server.aggregate()
#                 metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_tf1(
#                     server.get_model(),
#                     X_test,
#                     y_test_traffic,
#                     y_test_device,
#                     comm_round + 1,
#                     dataloader.label_encoders,
#                     task=task,
#                     model_name=model_name
#                 )

#             test_end_time = time.time()
#             test_time = test_end_time - test_start_time
#             round_train_time = time.time() - round_train_start_time - test_time
#             round_metrics = metrics.copy()
#             round_metrics.update({
#                 'model': model_name,
#                 'round': comm_round + 1,
#                 'train_time': round_train_time,
#                 'test_time': test_time,
#                 'test_per_sample': test_time / len(X_test),
#                 'train_memory_usage_MB': np.mean(round_train_metrics['memory_usage_MB']),
#                 'train_cpu_usage_percent': np.mean(round_train_metrics['cpu_usage_percent']),
#                 'train_energy_usage_cpu_sec': np.mean(round_train_metrics['energy_usage_cpu_sec'])
#             })
#             metrics_log.append(round_metrics)
#             print(f"\n{model_name} Metrics after Round {comm_round + 1}:")
#             table_data = [[k, f"{v:.4f}" if isinstance(v, (float, np.floating)) else v] for k, v in round_metrics.items()]
#             print(tabulate(table_data, headers=['Metric', 'Value'], tablefmt='grid'))

#         # Calculate total training time for the model
#         model_train_time = time.time() - model_train_start_time

#         # For ML models, generate and save ensemble results
#         if model_type == 'ml':
#             print(f"\nGenerating ensemble results for {model_name}")
#             metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_ml(
#                 server.get_model(),
#                 X_test,
#                 y_test_traffic,
#                 y_test_device,
#                 None,  # Final evaluation
#                 dataloader.label_encoders,
#                 multi_output=model_config['params']['multi_output'],
#                 task=task,
#                 model_name=model_name
#             )
#             save_ensemble_results(
#                 metrics,
#                 preds_traffic,
#                 preds_device,
#                 labels_traffic,
#                 labels_device,
#                 dataloader.label_encoders,
#                 model_name,
#                 model_config['params']['multi_output'],
#                 task
#             )
#             # Update metrics with final model training time and average client metrics
#             client_avg_metrics = {
#                 'train_memory_usage_MB': np.mean([m['memory_usage_MB'] for client in client_metrics.values() for m in client]),
#                 'train_cpu_usage_percent': np.mean([m['cpu_usage_percent'] for client in client_metrics.values() for m in client]),
#                 'train_energy_usage_cpu_sec': np.mean([m['energy_usage_cpu_sec'] for client in client_metrics.values() for m in client])
#             }
#             final_metrics = metrics.copy()
#             final_metrics.update({
#                 'model': model_name,
#                 'round': 'ensemble',
#                 'train_time': model_train_time,
#                 'test_time': test_time,
#                 'test_per_sample': test_time / len(X_test),
#                 'train_memory_usage_MB': client_avg_metrics['train_memory_usage_MB'],
#                 'train_cpu_usage_percent': client_avg_metrics['train_cpu_usage_percent'],
#                 'train_energy_usage_cpu_sec': client_avg_metrics['train_energy_usage_cpu_sec']
#             })
#             metrics_log.append(final_metrics)

#         # Print privacy budgets for DL models
#         if model_type == 'dl':
#             print(f"\nPrivacy Budgets for {model_name}:")
#             for entry in privacy_log:
#                 if entry['model'] == model_name:
#                     print(
#                         f"Client: {entry['client']}, "
#                         f"Round: {entry['communication_round']}, "
#                         f"Epoch: {entry['epoch']}, "
#                         f"Epsilon: {entry['epsilon']:.2f}"
#                     )

#         # Save final model
#         os.makedirs('models', exist_ok=True)
#         os.makedirs('confusion_matrices', exist_ok=True)
#         if model_type == 'dl':
#             save_pytorch_model(
#                 server.get_model(),
#                 dataloader.scaler,
#                 dataloader.label_encoders,
#                 feature_columns,
#                 f'models/{model_name}_final.pth',
#                 task=task
#             )
#             if args.multi_output or task == 'traffic':
#                 plot_confusion_matrix(labels_traffic, preds_traffic, f'confusion_matrices/{model_name}_traffic_final.png', dataloader.label_encoders['traffic'], 'Traffic Type')
#             if args.multi_output or task == 'device':
#                 plot_confusion_matrix(labels_device, preds_device, f'confusion_matrices/{model_name}_device_final.png', dataloader.label_encoders['device'], 'Device')
#         elif model_type == 'tf':
#             save_tf_model(server.get_model(), f'models/{model_name}_final.h5')
#             if args.multi_output or task == 'traffic':
#                 plot_confusion_matrix(labels_traffic, preds_traffic, f'confusion_matrices/{model_name}_traffic_final.png', dataloader.label_encoders['traffic'], 'Traffic Type')
#             if args.multi_output or task == 'device':
#                 plot_confusion_matrix(labels_device, preds_device, f'confusion_matrices/{model_name}_device_final.png', dataloader.label_encoders['device'], 'Device')
#         elif model_type == 'ml':
#             for client_name, client in clients.items():
#                 if model_config['params']['multi_output']:
#                     save_ml_model(
#                         client.get_model(),
#                         dataloader.scaler,
#                         dataloader.label_encoders,
#                         feature_columns,
#                         f'models/{model_name}_{client_name}_final.joblib',
#                         multi_output=True
#                     )
#                 else:
#                     if task == 'traffic' or model_name.endswith('_traffic'):
#                         save_ml_model(
#                             client.get_model(task='traffic'),
#                             dataloader.scaler,
#                             dataloader.label_encoders,
#                             feature_columns,
#                             f'models/{model_name}_{client_name}_traffic_final.joblib',
#                             multi_output=False
#                         )
#                     if task == 'device' or model_name.endswith('_device'):
#                         save_ml_model(
#                             client.get_model(task='device'),
#                             dataloader.scaler,
#                             dataloader.label_encoders,
#                             feature_columns,
#                             f'models/{model_name}_{client_name}_device_final.joblib',
#                             multi_output=False
#                         )
#             if args.multi_output or task == 'traffic':
#                 plot_confusion_matrix(labels_traffic, preds_traffic, f'confusion_matrices/{model_name}_traffic_final.png', dataloader.label_encoders['traffic'], 'Traffic Type')
#             if (args.multi_output or task == 'device') and preds_device is not None:
#                 plot_confusion_matrix(labels_device, preds_device, f'confusion_matrices/{model_name}_device_final.png', dataloader.label_encoders['device'], 'Device')

#     # Save metrics to CSV
#     os.makedirs('metrics', exist_ok=True)
#     metrics_df = pd.DataFrame(metrics_log)
#     metrics_df.to_csv('metrics_final.csv', index=False)
#     print("\nFinal metrics saved to 'metrics_final.csv'")

#     # Save privacy budget to CSV
#     if privacy_log:
#         save_privacy_budget(privacy_log)

No GPU detected. Running TensorFlow on CPU.
TensorFlow version: 2.19.0


In [4]:
if __name__ == "__main__":
    csv_file = r"F:/Datasets_Project/NIM LAB IoT Dataset 2025/slowloriss/small/processed/NIMLABIoT_processed.csv"
    X_scaled, y_device_encoded, y_attack_encoded, le_device, le_attack, feature_columns = load_and_preprocess_data(csv_file)
    print(f"\nNumber of features used: {len(feature_columns)}")
    print("Features used:", feature_columns)
    dataloader = DatasetLoader(X_scaled, y_device_encoded, y_attack_encoded, le_device, le_attack,
                              num_clients=args.num_clients, sample_rate=args.sample_rate)
    dataloader.print_client_label_distribution()
    train_loader = dataloader.get_train_data()
    test_loader = dataloader.get_test_data()
    X_train, y_train_traffic, y_train_device, X_test, y_test_traffic, y_test_device = dataloader.get_ml_data()
    input_shape = X_scaled.shape[1]
    num_classes_traffic = len(np.unique(y_attack_encoded))
    num_classes_device = len(np.unique(y_device_encoded))
    clients_data = dataloader.clients_data
    local_epochs = max(1, args.epochs // args.communication_rounds)

    models = []
    if args.model_type in ['dl', 'both']:
        if args.multi_output:
            models.extend([
                {'name': 'cnn', 'type': 'dl', 'class': MultiOutputCNN, 'server': DLServer, 'task': 'both'},
                {'name': 'lstm', 'type': 'dl', 'class': LSTMModel, 'server': DLServer, 'task': 'both'},
                {'name': 'bilstm', 'type': 'dl', 'class': BiLSTM, 'server': DLServer, 'task': 'both'},
                {'name': 'tf1cnn', 'type': 'tf', 'class': TF1DCNN, 'server': TF1Server, 'task': 'both'}
            ])
        else:
            models.extend([
                {'name': 'cnn_traffic', 'type': 'dl', 'class': MultiOutputCNN, 'server': DLServer, 'task': 'traffic'},
                {'name': 'cnn_device', 'type': 'dl', 'class': MultiOutputCNN, 'server': DLServer, 'task': 'device'},
                {'name': 'lstm_traffic', 'type': 'dl', 'class': LSTMModel, 'server': DLServer, 'task': 'traffic'},
                {'name': 'lstm_device', 'type': 'dl', 'class': LSTMModel, 'server': DLServer, 'task': 'device'},
                {'name': 'bilstm_traffic', 'type': 'dl', 'class': BiLSTM, 'server': DLServer, 'task': 'traffic'},
                {'name': 'bilstm_device', 'type': 'dl', 'class': BiLSTM, 'server': DLServer, 'task': 'device'},
                {'name': 'tf1cnn_traffic', 'type': 'tf', 'class': TF1DCNN, 'server': TF1Server, 'task': 'traffic'},
                {'name': 'tf1cnn_device', 'type': 'tf', 'class': TF1DCNN, 'server': TF1Server, 'task': 'device'}
            ])
    if args.model_type in ['ml', 'both']:
        if args.multi_output:
            models.extend([
                {'name': 'xgboost', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'xgboost', 'multi_output': True}},
                {'name': 'rf', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'rf', 'multi_output': True}},
                {'name': 'dt', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'dt', 'multi_output': True}}
            ])
        else:
            models.extend([
                {'name': 'xgboost_traffic', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'xgboost_traffic', 'multi_output': False}},
                {'name': 'xgboost_device', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'xgboost_device', 'multi_output': False}},
                {'name': 'rf_traffic', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'rf_traffic', 'multi_output': False}},
                {'name': 'rf_device', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'rf_device', 'multi_output': False}},
                {'name': 'dt_traffic', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'dt_traffic', 'multi_output': False}},
                {'name': 'dt_device', 'type': 'ml', 'class': MLModel, 'server': MLServer, 'params': {'model_type': 'dt_device', 'multi_output': False}}
            ])

    metrics_log = []
    privacy_log = []

    for model_config in models:
        model_name = model_config['name']
        model_type = model_config['type']
        task = model_config.get('task', 'both')
        print(f"\nTraining model: {model_name}")
        model_train_start_time = time.time()

        if model_type == 'dl':
            server = model_config['server'](
                model_class=model_config['class'],
                input_size=input_shape,
                num_classes_traffic=num_classes_traffic,
                num_classes_device=num_classes_device,
                task=task
            )
            clients = {
                client_name: DLClient(
                    model_class=model_config['class'],
                    data_loader=client_data['dl_loader'],
                    input_size=input_shape,
                    num_classes_traffic=num_classes_traffic,
                    num_classes_device=num_classes_device,
                    y_train_traffic=client_data['y_traffic'],
                    y_train_device=client_data['y_device'],
                    task=task
                )
                for client_name, client_data in clients_data.items()
            }
        elif model_type == 'tf':
            server = model_config['server'](
                input_shape=input_shape,
                num_classes_traffic=num_classes_traffic,
                num_classes_device=num_classes_device,
                task=task
            )
            clients = {
                client_name: TF1Client(
                    X=client_data['X'],
                    y_traffic=client_data['y_traffic'],
                    y_device=client_data['y_device'],
                    input_shape=input_shape,
                    num_classes_traffic=num_classes_traffic,
                    num_classes_device=num_classes_device,
                    task=task
                )
                for client_name, client_data in clients_data.items()
            }
        elif model_type == 'ml':
            server = model_config['server'](**model_config['params'])
            clients = {
                client_name: MLClient(
                    model_type=model_config['params']['model_type'],
                    X=client_data['X'],
                    y_traffic=client_data['y_traffic'],
                    y_device=client_data['y_device'],
                    multi_output=model_config['params']['multi_output']
                )
                for client_name, client_data in clients_data.items()
            }

        client_metrics = {}  # Store per-client training metrics
        for comm_round in range(args.communication_rounds):
            print(f"\n{model_name} Communication Round {comm_round + 1}/{args.communication_rounds}")
            round_train_start_time = time.time()
            round_train_metrics = {
                'memory_usage_MB': [],
                'cpu_usage_percent': [],
                'energy_usage_cpu_sec': []
            }

            for client_name, client in clients.items():
                print(f"\nTraining {client_name}")
                if model_type == 'dl':
                    client.set_parameters(server.get_model().parameters())
                    epsilons, train_metrics = client.train(local_epochs, client_name)
                    server.add_client(client.get_parameters(), len(client.data_loader.dataset))
                    for epoch, epsilon in enumerate(epsilons, 1):
                        privacy_log.append({
                            'model': model_name,
                            'client': client_name,
                            'communication_round': comm_round + 1,
                            'epoch': epoch,
                            'epsilon': epsilon
                        })
                elif model_type == 'tf':
                    client.set_parameters(server.get_model().get_weights())
                    train_metrics = client.train(local_epochs, client_name)
                    server.add_client(client.get_parameters(), len(client.X))
                elif model_type == 'ml':
                    train_metrics = client.train(local_epochs, client_name)
                    if model_config['params']['multi_output']:
                        server.add_client(client.get_model(), client_name, client.X, client.y_traffic, client.y_device)
                    else:
                        if task == 'traffic' or model_name.endswith('_traffic'):
                            server.add_client(client.get_model(task='traffic'), client_name, client.X, client.y_traffic, None)
                        if task == 'device' or model_name.endswith('_device'):
                            server.add_client(client.get_model(task='device'), client_name, client.X, None, client.y_device)
                client_metrics[client_name] = client_metrics.get(client_name, []) + [train_metrics]
                for key in round_train_metrics:
                    round_train_metrics[key].append(train_metrics[key])

            # Aggregate and evaluate
            test_start_time = time.time()
            if model_type == 'ml' and task == 'both':
                attack_preds, device_preds, attack_probs, device_probs = server.aggregate(X_test)
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_ml(
                    server.get_model(),
                    X_test,
                    y_test_traffic,
                    y_test_device,  # Fixed typo from previous version
                    comm_round + 1,
                    dataloader.label_encoders,
                    multi_output=model_config['params']['multi_output'],
                    task=task,
                    model_name=model_name
                )
            elif model_type == 'ml':
                attack_preds, device_preds, attack_probs, device_probs = server.aggregate(X_test)
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_ml(
                    server.get_model(),
                    X_test,
                    y_test_traffic,
                    y_test_device,
                    comm_round + 1,
                    dataloader.label_encoders,
                    multi_output=model_config['params']['multi_output'],
                    task=task,
                    model_name=model_name
                )
            elif model_type == 'dl':
                server.aggregate()
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_dl(
                    server.get_model(),
                    args.device,
                    test_loader,
                    comm_round + 1,
                    dataloader.label_encoders,
                    task=task,
                    model_name=model_name
                )
            elif model_type == 'tf':
                server.aggregate()
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_tf1(
                    server.get_model(),
                    X_test,
                    y_test_traffic,
                    y_test_device,
                    comm_round + 1,
                    dataloader.label_encoders,
                    task=task,
                    model_name=model_name
                )

            test_end_time = time.time()
            test_time = test_end_time - test_start_time
            round_train_time = time.time() - round_train_start_time - test_time
            round_metrics = metrics.copy()
            round_metrics.update({
                'model': model_name,
                'round': comm_round + 1,
                'train_time': round_train_time,
                'test_time': test_time,
                'test_per_sample': test_time / len(X_test),
                'train_memory_usage_MB': np.mean(round_train_metrics['memory_usage_MB']),
                'train_cpu_usage_percent': np.mean(round_train_metrics['cpu_usage_percent']),
                'train_energy_usage_cpu_sec': np.mean(round_train_metrics['energy_usage_cpu_sec'])
            })
            metrics_log.append(round_metrics)
            print(f"\n{model_name} Metrics after Round {comm_round + 1}:")
            table_data = [[k, f"{v:.4f}" if isinstance(v, (float, np.floating)) else v] for k, v in round_metrics.items()]
            print(tabulate(table_data, headers=['Metric', 'Value'], tablefmt='grid'))

        # Calculate total training time for the model
        model_train_time = time.time() - model_train_start_time

        # Final evaluation of the aggregated server model
        print(f"\nGenerating final aggregated results for {model_name}")
        try:
            if model_type == 'dl':
                server.aggregate()  # Ensure final aggregation
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_dl(
                    server.get_model(),
                    args.device,
                    test_loader,
                    None,  # Final evaluation
                    dataloader.label_encoders,
                    task=task,
                    model_name=model_name
                )
                save_ensemble_results(
                    metrics,
                    preds_traffic,
                    preds_device,
                    labels_traffic,
                    labels_device,
                    dataloader.label_encoders,
                    model_name,
                    args.multi_output,
                    task
                )
            elif model_type == 'tf':
                server.aggregate()  # Ensure final aggregation
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_tf1(
                    server.get_model(),
                    X_test,
                    y_test_traffic,
                    y_test_device,
                    None,  # Final evaluation
                    dataloader.label_encoders,
                    task=task,
                    model_name=model_name
                )
                save_ensemble_results(
                    metrics,
                    preds_traffic,
                    preds_device,
                    labels_traffic,
                    labels_device,
                    dataloader.label_encoders,
                    model_name,
                    args.multi_output,
                    task
                )
            elif model_type == 'ml':
                attack_preds, device_preds, attack_probs, device_probs = server.aggregate(X_test)
                metrics, (preds_traffic, preds_device), (labels_traffic, labels_device), (probs_traffic, probs_device) = evaluate_ml(
                    server.get_model(),
                    X_test,
                    y_test_traffic,
                    y_test_device,
                    None,  # Final evaluation
                    dataloader.label_encoders,
                    multi_output=model_config['params']['multi_output'],
                    task=task,
                    model_name=model_name
                )
                save_ensemble_results(
                    metrics,
                    preds_traffic,
                    preds_device,
                    labels_traffic,
                    labels_device,
                    dataloader.label_encoders,
                    model_name,
                    model_config['params']['multi_output'],
                    task
                )
        except Exception as e:
            print(f"Error during final aggregation/evaluation for {model_name}: {e}")
            metrics = {
                'model': model_name,
                'round': 'final',
                'error': str(e)
            }
            preds_traffic, preds_device, labels_traffic, labels_device, probs_traffic, probs_device = [], [], [], [], [], []

        # Update metrics with final model training time and average client metrics
        client_avg_metrics = {
            'train_memory_usage_MB': np.mean([m['memory_usage_MB'] for client in client_metrics.values() for m in client]) if client_metrics else 0.0,
            'train_cpu_usage_percent': np.mean([m['cpu_usage_percent'] for client in client_metrics.values() for m in client]) if client_metrics else 0.0,
            'train_energy_usage_cpu_sec': np.mean([m['energy_usage_cpu_sec'] for client in client_metrics.values() for m in client]) if client_metrics else 0.0
        }
        final_metrics = metrics.copy()
        final_metrics.update({
            'model': model_name,
            'round': 'final',
            'train_time': model_train_time,
            'test_time': test_time,
            'test_per_sample': test_time / len(X_test) if X_test.size > 0 else 0.0,
            'train_memory_usage_MB': client_avg_metrics['train_memory_usage_MB'],
            'train_cpu_usage_percent': client_avg_metrics['train_cpu_usage_percent'],
            'train_energy_usage_cpu_sec': client_avg_metrics['train_energy_usage_cpu_sec']
        })
        metrics_log.append(final_metrics)

        # Print privacy budgets for DL models
        if model_type == 'dl':
            print(f"\nPrivacy Budgets for {model_name}:")
            for entry in privacy_log:
                if entry['model'] == model_name:
                    print(
                        f"Client: {entry['client']}, "
                        f"Round: {entry['communication_round']}, "
                        f"Epoch: {entry['epoch']}, "
                        f"Epsilon: {entry['epsilon']:.2f}"
                    )

        # Save final model
        os.makedirs('models', exist_ok=True)
        os.makedirs('confusion_matrices', exist_ok=True)
        if model_type == 'dl':
            save_pytorch_model(
                server.get_model(),
                dataloader.scaler,
                dataloader.label_encoders,
                feature_columns,
                f'models/{model_name}_final.pth',
                task=task
            )
            if (args.multi_output or task == 'traffic') and len(preds_traffic) > 0:
                plot_confusion_matrix(labels_traffic, preds_traffic, f'confusion_matrices/{model_name}_traffic_final.png', dataloader.label_encoders['traffic'], 'Traffic Type')
            if (args.multi_output or task == 'device') and len(preds_device) > 0:
                plot_confusion_matrix(labels_device, preds_device, f'confusion_matrices/{model_name}_device_final.png', dataloader.label_encoders['device'], 'Device')
        elif model_type == 'tf':
            save_tf_model(server.get_model(), f'models/{model_name}_final.h5')
            if (args.multi_output or task == 'traffic') and len(preds_traffic) > 0:
                plot_confusion_matrix(labels_traffic, preds_traffic, f'confusion_matrices/{model_name}_traffic_final.png', dataloader.label_encoders['traffic'], 'Traffic Type')
            if (args.multi_output or task == 'device') and len(preds_device) > 0:
                plot_confusion_matrix(labels_device, preds_device, f'confusion_matrices/{model_name}_device_final.png', dataloader.label_encoders['device'], 'Device')
        elif model_type == 'ml':
            for client_name, client in clients.items():
                if model_config['params']['multi_output']:
                    save_ml_model(
                        client.get_model(),
                        dataloader.scaler,
                        dataloader.label_encoders,
                        feature_columns,
                        f'models/{model_name}_{client_name}_final.joblib',
                        multi_output=True
                    )
                else:
                    if task == 'traffic' or model_name.endswith('_traffic'):
                        save_ml_model(
                            client.get_model(task='traffic'),
                            dataloader.scaler,
                            dataloader.label_encoders,
                            feature_columns,
                            f'models/{model_name}_{client_name}_traffic_final.joblib',
                            multi_output=False
                        )
                    if task == 'device' or model_name.endswith('_device'):
                        save_ml_model(
                            client.get_model(task='device'),
                            dataloader.scaler,
                            dataloader.label_encoders,
                            feature_columns,
                            f'models/{model_name}_{client_name}_device_final.joblib',
                            multi_output=False
                        )
            if (args.multi_output or task == 'traffic') and len(preds_traffic) > 0:
                plot_confusion_matrix(labels_traffic, preds_traffic, f'confusion_matrices/{model_name}_traffic_final.png', dataloader.label_encoders['traffic'], 'Traffic Type')
            if (args.multi_output or task == 'device') and len(preds_device) > 0 and preds_device is not None:
                plot_confusion_matrix(labels_device, preds_device, f'confusion_matrices/{model_name}_device_final.png', dataloader.label_encoders['device'], 'Device')

    # Save metrics to CSV
    os.makedirs('metrics', exist_ok=True)
    metrics_df = pd.DataFrame(metrics_log)
    metrics_df.to_csv('metrics_final.csv', index=False)
    print("\nFinal metrics saved to 'metrics_final.csv'")

    # Save privacy budget to CSV
    if privacy_log:
        save_privacy_budget(privacy_log)

Loaded NIMLABIoT_processed.csv with 1415685 rows and 76 columns.

Null values per column:
Pck_size           0
payload_bytes      0
TCP_mss_values     0
HTTP_status        0
DHCP_options       0
                  ..
Payload_Len_Q1     0
Payload_Len_Q3     0
Payload_Len_IQR    0
Traffic Type       0
Label              0
Length: 76, dtype: int64
After dropping NaN rows, dataset has 1415685 rows.

Number of duplicate rows: 0

Column data types:
Pck_size            int64
payload_bytes       int64
TCP_mss_values      int64
HTTP_status         int64
DHCP_options        int64
                    ...  
Payload_Len_Q1      int64
Payload_Len_Q3      int64
Payload_Len_IQR     int64
Traffic Type       object
Label              object
Length: 76, dtype: object

Device type class distribution before dropping rare classes:
Label
Amcrest         240000
Nestcam         195432
Nestmini        193132
SamsungTV       186911
Smartplug       184014
Powerstrip      180904
RingDoorbell    160820
Coffeemaker  

C:\Users\danie\AppData\Local\Temp\ipykernel_8768\3359007943.py:84: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = df.groupby('Label', group_keys=False).apply(



Sampled Label value counts (50% of each):
Label
Amcrest         96000
Nestcam         78173
Nestmini        77253
SamsungTV       74764
Smartplug       73606
Powerstrip      72362
RingDoorbell    64328
Coffeemaker     29789
Name: count, dtype: int64

Original rows: 1415685, Sampled rows: 566275

Non-numeric columns: []

SMOTE disabled. Using original dataset.
Dataset has 566275 rows.
Removing 11 zero-variance features: ['STUN_method', 'MQTT_type', 'TCP_URG', 'TCP_CWR', 'DNS_opcode', 'DNS_z', 'DNS_nscount', 'Direction_ratio', 'TLS_Cipher_Suites_Len', 'TLS_Server', 'NTP_Interval']

Number of features used: 63
Features used: ['Pck_size', 'payload_bytes', 'TCP_mss_values', 'HTTP_status', 'DHCP_options', 'IP_version', 'IP_proto', 'IP_chksum', 'IP_add_count', 'TCP_sport', 'TCP_seq', 'TCP_dataofs', 'TCP_FIN', 'TCP_SYN', 'TCP_RST', 'TCP_PSH', 'TCP_ACK', 'TCP_chksum', 'TCP_window_scaling', 'UDP_sport', 'UDP_chksum', 'ICMP_type', 'ICMP_chksum', 'ICMP_seq', 'BOOTP_hlen', 'BOOTP_options', 'DNS_id

C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\opacus\privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\opacus\privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
C:\Users\danie\anaconda3\envs\HybridDualDistillation\lib\site-packages\opacus\privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production


cnn Communication Round 1/10

Training client_0


...

Training client_4 TF1 Epoch: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [04:23<00:00, 26.35s/it]

Client client_4 trained for 10 epochs. Memory: 970.61 MB, CPU: 23.52%, Energy: 6198.23 CPU-sec


traffic unique classes: [0 1 2 3]
traffic y_probs shape: (113255, 4)
traffic expected classes: 4
traffic AUC (multiclass): 0.9999662521680852
device unique classes: [0 1 2 3 4 5 6 7]
device y_probs shape: (113255, 8)
device expected classes: 8


06/30/2025 01:23:14:WARNING:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 


device AUC (multiclass): 0.9999203786029502

tf1cnn Metrics after Round 10:
+----------------------------+-----------+
| Metric                     | Value     |
+============================+===========+
| traffic_precision          | 0.9974    |
+----------------------------+-----------+
| traffic_recall             | 0.9974    |
+----------------------------+-----------+
| traffic_f1                 | 0.9974    |
+----------------------------+-----------+
| traffic_kappa              | 0.9963    |
+----------------------------+-----------+
| traffic_mcc                | 0.9963    |
+----------------------------+-----------+
| traffic_balanced_accuracy  | 0.9976    |
+----------------------------+-----------+
| traffic_hamming_loss       | 0.0026    |
+----------------------------+-----------+
| traffic_tnr                | 0.9991    |
+----------------------------+-----------+
| traffic_specificity        | 0.9991    |
+----------------------------+-----------+
| traffic_tpr        